# Beyond LangGraph and CrewAI: The Agent Framework Landscape, Hands-On

> **Who is this for:** you have built a small agent (or read the LangGraph / CrewAI notebooks) and now you see twenty other framework names on job ads and blog posts. You want to know what each one is, when to pick it, and what its code actually looks like.
>
> **What you will learn:**
> - the handful of **ideas every agent framework shares** (agent, tool, loop, handoff, state, tracing) and the "families" frameworks fall into
> - a short, honest profile of **20+ Python frameworks**: Microsoft Agent Framework (and AutoGen / AG2 / Semantic Kernel), OpenAI Agents SDK, Google ADK, Claude Agent SDK, PydanticAI, smolagents, LlamaIndex, Haystack, Agno, Strands, DSPy, Deep Agents, Letta, CAMEL, MetaGPT, BeeAI, Langroid, Atomic Agents and Swarm
> - the **TypeScript** options (Mastra, Vercel AI SDK, LangGraph.js, OpenAI Agents JS) and the **managed cloud platforms** (AWS, Azure, Google Cloud, OpenAI)
> - **the same task written in 11 real frameworks**, run on your laptop with **scripted stand-in models** (no API key)
> - a big **comparison table**, a **"how to choose" flowchart**, and a tiny chooser function
>
> **Prerequisites:** [Agent fundamentals](01_agent_fundamentals.ipynb) (the agent loop and tools). Helpful: [LangGraph hands-on](12_langgraph_hands_on.ipynb), [CrewAI and multi-agent frameworks hands-on](13_crewai_and_multi_agent_frameworks_hands_on.ipynb), [MCP and tools](05_mcp_and_tools.ipynb).
>
> **Time needed:** about 1.5-2 hours to read. The notebook itself runs in well under a minute.
>
> **Runs on:** CPU, **no API key, no internet**. Every framework cell is optional: if a framework is not installed, the cell prints `skipped: needs <package>` and the notebook keeps going.

**Note on dates.** Agent frameworks change faster than almost anything else in AI. Names, versions and "who merged with whom" below are written **as of late 2026**. When a project was renamed, merged or deprecated, this notebook says so. Always check the official docs (linked at the end) before you start a real project.

### Optional installs

The notebook runs with only the standard repo packages (numpy, pandas). To see the **real** framework code run, install some or all of these into a **separate virtual environment** (they pull in many dependencies, and some pin different versions of the same libraries):

```bash
python -m venv .venv-frameworks && source .venv-frameworks/bin/activate
pip install pydantic-ai-slim autogen-agentchat autogen-ext agent-framework-core agent-framework-orchestrations openai-agents google-adk smolagents llama-index-core haystack-ai agno strands-agents dspy
```

You can also install just one, e.g. `pip install pydantic-ai-slim`. The notebook was last executed with: pydantic-ai-slim 2.68, autogen-agentchat / autogen-ext 0.7.5, agent-framework-core 1.26, openai-agents 0.24, google-adk 2.14, smolagents 1.26, llama-index-core 0.14, haystack-ai 3.5, agno 3.2, strands-agents 1.64, dspy 3.5 (all on Python 3.12, November 2026). Newer versions may rename things; each demo catches that and prints a short message instead of crashing.

Frameworks shown **only as code** (not run here), with their install names for later: `crewai`, `langgraph`, `semantic-kernel`, `ag2`, `claude-agent-sdk`, `deepagents`, `letta-client`, `camel-ai`, `metagpt`, `beeai-framework`, `langroid`, `atomic-agents`.

## 0. Setup

Three things happen here:
1. We turn off **telemetry** (some frameworks send anonymous usage pings by default) and **tracing exports**, so nothing tries to reach the internet.
2. We check which frameworks are installed.
3. We define a small helper, `run_demo`, that runs one framework demo and records the result. If the framework is missing it prints `skipped: needs ...`.

In [1]:
import os, sys, json, random, inspect, logging, warnings, importlib.util, importlib.metadata

# --- keep every framework offline and quiet (these only matter if the framework is installed) ---
os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")          # PydanticAI: no start-up banner
os.environ.setdefault("AGNO_TELEMETRY", "false")             # Agno: no usage pings
os.environ.setdefault("HAYSTACK_TELEMETRY_ENABLED", "False") # Haystack: no usage pings
os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")  # OpenAI Agents SDK: do not upload traces
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")# DSPy uses LiteLLM: do not download a price list
os.environ.setdefault("OTEL_SDK_DISABLED", "true")           # OpenTelemetry: do not export spans
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)       # hide framework log chatter (e.g. "token usage missing" notices)

import numpy as np
import pandas as pd
random.seed(0); np.random.seed(0)      # fixed seed (nothing here is random, but it is a good habit)
pd.set_option("display.max_colwidth", 80)

# label -> (module to import, PyPI distribution name, pip install command)
FRAMEWORKS = {
    "PydanticAI":              ("pydantic_ai",       "pydantic-ai-slim",   "pydantic-ai-slim"),
    "AutoGen AgentChat":       ("autogen_agentchat", "autogen-agentchat",  "autogen-agentchat autogen-ext"),
    "Microsoft Agent Framework": ("agent_framework", "agent-framework-core", "agent-framework-core agent-framework-orchestrations"),
    "OpenAI Agents SDK":       ("agents",            "openai-agents",      "openai-agents"),
    "Google ADK":              ("google.adk",        "google-adk",         "google-adk"),
    "smolagents":              ("smolagents",        "smolagents",         "smolagents"),
    "LlamaIndex":              ("llama_index.core",  "llama-index-core",   "llama-index-core"),
    "Haystack":                ("haystack",          "haystack-ai",        "haystack-ai"),
    "Agno":                    ("agno",              "agno",               "agno"),
    "Strands Agents":          ("strands",           "strands-agents",     "strands-agents"),
    "DSPy":                    ("dspy",              "dspy",               "dspy"),
}

def installed_version(dist):
    try:
        return importlib.metadata.version(dist)
    except importlib.metadata.PackageNotFoundError:
        return None

def module_found(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except (ImportError, ValueError):
        return False

status = pd.DataFrame([
    {"framework": k, "pip install": v[2],
     "installed version": (installed_version(v[1]) if module_found(v[0]) else None) or "not installed"}
    for k, v in FRAMEWORKS.items()])
print(f"Python {sys.version.split()[0]}")
status

Python 3.12.3


,framework,pip install,installed version
0,PydanticAI,pydantic-ai-slim,2.68.0
1,AutoGen AgentChat,autogen-agentchat autogen-ext,0.7.5
2,Microsoft Agent Framework,agent-framework-core agent-framework-orchestrations,1.26.0
3,OpenAI Agents SDK,openai-agents,0.24.0
4,Google ADK,google-adk,2.14.1
5,smolagents,smolagents,1.26.0
6,LlamaIndex,llama-index-core,0.14.25
7,Haystack,haystack-ai,3.5.0
8,Agno,agno,3.2.0
9,Strands Agents,strands-agents,1.64.0


In [2]:
RESULTS = []   # one row per demo: which framework, which task, did it run, what did it answer

async def run_demo(framework, task, demo):
    '''Run one framework demo. `demo` is a function (sync or async) that returns the final answer.
    - framework not installed  -> print "skipped: needs ..." (ImportError)
    - installed but the API changed in a newer version -> print a short note instead of crashing'''
    pip_name = FRAMEWORKS.get(framework, (None, None, framework))[2]
    try:
        answer = demo()
        if inspect.isawaitable(answer):
            answer = await answer
        RESULTS.append({"framework": framework, "task": task, "status": "ran", "final answer": str(answer)})
        print(f"[{framework}] final answer -> {answer}")
    except ImportError:
        RESULTS.append({"framework": framework, "task": task, "status": "skipped", "final answer": ""})
        print(f"skipped: needs {pip_name}   (pip install {pip_name})")
    except Exception as e:   # frameworks change fast; keep the notebook running
        RESULTS.append({"framework": framework, "task": task, "status": "api-changed", "final answer": ""})
        print(f"[{framework}] did not run with your installed version: {type(e).__name__}: {str(e)[:200]}")
        print("  (APIs in this space change often - compare with the official docs linked at the end.)")

print("helper ready")

helper ready


**What just happened?** We listed the frameworks this notebook can run and their installed versions. Anything marked "not installed" will simply be skipped later. The helper `run_demo` is how every framework demo is called, so one missing package never breaks the notebook.

A note on `await`: many frameworks are **async** (their `run` functions are coroutines, which you wait for with `await`). Jupyter lets you write `await` directly in a cell, so you will see `await run_demo(...)` below. In a normal `.py` script you would wrap it in `asyncio.run(...)`.

## 1. Why are there so many agent frameworks?

**What it is.** An **agent framework** is a library that gives you ready-made pieces for building an LLM agent: a way to describe the agent, plug in **tools** (functions the model may call), run the **agent loop** (model -> tool -> model -> ... -> answer), remember **state**, and **trace** what happened.

**Simple analogy.** Web frameworks: Flask, Django, FastAPI and Express all do "receive request, return response". They differ in how much they decide for you, which language they use, and what extras they ship. Agent frameworks are the same: they all wrap the same loop, then add different opinions on top.

**How it works (the shared core).** Strip any framework down and you find this loop:

```mermaid
flowchart LR
    U[User question] --> M[Model call]
    M -->|wants a tool| T[Run the tool in Python]
    T -->|tool result added to messages| M
    M -->|final text| A[Answer]
```

Everything else is an "extra" built around that loop. The extras are where frameworks differ:

| Extra | Plain meaning | Example |
|---|---|---|
| **Multi-agent** | several agents with different jobs that pass work between them | researcher -> writer |
| **Handoff** | one agent gives the conversation to another agent | triage agent hands to billing agent |
| **Orchestration** | the rule for who runs next: fixed order, graph, manager agent, or the model decides | sequential, group chat, graph |
| **State / memory** | what the agent remembers inside a run and between runs | chat history, user profile |
| **Human-in-the-loop (HITL)** | the agent pauses so a person can approve or edit | "approve this refund?" |
| **Guardrail** | a check that blocks bad input or output | block requests for medical dosage |
| **Tracing** | a recorded timeline of every model call and tool call | OpenTelemetry spans |
| **MCP** | Model Context Protocol: a standard way to plug in tools from a separate server | a GitHub MCP server |
| **A2A** | Agent2Agent protocol: a standard way for agents built with different frameworks to talk over the network | your ADK agent calls a partner's agent |
| **Durable execution** | the run survives crashes and can resume later from a saved checkpoint | a 3-day approval workflow |

### The families

Most frameworks fall into a few "families". Knowing the family tells you most of what you need.

| Family | Core idea | Examples |
|---|---|---|
| **Graph / workflow** | you draw the steps and arrows yourself; the model fills in the boxes | LangGraph, LlamaIndex Workflows, Microsoft Agent Framework workflows, ADK 2.0 workflows, Haystack pipelines |
| **Role-playing crews** | you describe agents as job roles with goals; the framework runs the team | CrewAI, CAMEL, MetaGPT, ChatDev |
| **Conversation / group chat** | agents talk in a shared chat; a rule picks the next speaker | AutoGen / AG2, Microsoft Agent Framework group chat |
| **Thin SDKs with handoffs** | a small set of primitives (agent, tool, handoff, guardrail), close to the model API | OpenAI Agents SDK, Google ADK, Strands, Agno, Swarm (old) |
| **Type-safe / validation-first** | inputs and outputs are typed Python objects; errors are caught early | PydanticAI, Atomic Agents |
| **Code-acting** | the model writes small Python programs as its actions instead of JSON tool calls | smolagents CodeAgent |
| **Programming + optimizing** | you declare what goes in and out; the framework writes and tunes the prompts | DSPy |
| **Stateful agent servers** | the agent is a long-lived service with its own memory database | Letta (formerly MemGPT) |
| **Agent harnesses** | a ready-made "do long tasks" agent with planning, files and sub-agents built in | Claude Agent SDK, LangChain Deep Agents, OpenAI Agents SDK sandbox agents |

**When to use a framework at all / when not to.** If your task is one or two model calls in a fixed order, plain code is clearer (see the baseline in section 2). Reach for a framework when you need several of the extras above (tracing, retries, handoffs, HITL, durable state) and do not want to build them yourself.

## 2. The shared toy task, and a no-framework baseline

We will use the same two tasks everywhere, so you can compare like with like:

1. **Calculator agent.** Question: *"A school orders 17 boxes with 23 pencils in each. How many pencils is that?"* The agent has one tool, `multiply(a, b)`. A good run: the model asks for `multiply(17, 23)`, Python runs it, the model reads `391` and answers.
2. **Research -> write handoff** (section 6). A *researcher* agent collects notes, then a *writer* agent turns the notes into a short paragraph.

**What is a "scripted stand-in model"?** Real agents call an LLM over the internet. Here we replace the LLM with a tiny **scripted model**: a few lines of Python that always reply the way a real model would (first "call multiply with 17 and 23", then "the answer is ..."). Every framework has a slot where the model plugs in, and most ship an official test double for exactly this purpose (`TestModel`, `ReplayChatCompletionClient`, `ScriptedModel`, `DummyLM`, `MockFunctionCallingLLM`, ...). So the framework code you see is **real**; only the "brain" is fake. Important: the scripted model builds its final sentence from the **tool result the framework actually gave it**, so if the framework did not run the tool, you would see it.

Here is the loop with **no framework at all**, so you can see what every framework automates.

In [3]:
QUESTION = "A school orders 17 boxes with 23 pencils in each. How many pencils is that?"

def multiply(a: float, b: float) -> float:
    '''Multiply two numbers.'''
    return a * b

def scripted_llm(messages):
    '''Stand-in for a real LLM. Turn 1: ask for the tool. Turn 2: answer using the tool result.'''
    tool_results = [m for m in messages if m["role"] == "tool"]
    if not tool_results:
        return {"role": "assistant", "tool_call": {"name": "multiply", "args": {"a": 17, "b": 23}}}
    return {"role": "assistant", "content": f"That is {tool_results[-1]['content']} pencils."}

def plain_agent_loop(question, tools, llm, max_steps=5):
    messages = [{"role": "system", "content": "Use tools for maths."},
                {"role": "user", "content": question}]
    for step in range(max_steps):                       # a step limit stops endless loops
        reply = llm(messages)
        messages.append(reply)
        if "tool_call" in reply:                        # the model wants a tool
            call = reply["tool_call"]
            result = tools[call["name"]](**call["args"])   # WE run the tool, not the model
            messages.append({"role": "tool", "name": call["name"], "content": str(result)})
            print(f"step {step}: model asked for {call['name']}{tuple(call['args'].values())} -> {result}")
        else:
            print(f"step {step}: model answered")
            return reply["content"], messages
    return "gave up", messages

answer, trace = plain_agent_loop(QUESTION, {"multiply": multiply}, scripted_llm)
print("\nFinal answer:", answer)
RESULTS.append({"framework": "plain Python (no framework)", "task": "calculator", "status": "ran", "final answer": answer})

step 0: model asked for multiply(17, 23) -> 391
step 1: model answered

Final answer: That is 391 pencils.


**What just happened?** In about 20 lines we built a working tool-calling agent. Notice what we had to do ourselves: keep the message list, detect tool calls, run the tool, add the result, limit the steps. We did **not** get: retries, tracing, typed outputs, streaming, multi-agent handoffs, saving state to a database, or human approval. Those are what frameworks sell. Keep this picture in mind: every framework below is "this loop plus extras".

## 3. Python code-first frameworks

For each framework: **what it is**, an **analogy**, **how it works**, **when to use / not use**, **strengths and weaknesses**, and code. When the framework can run offline, the code cell is a **real run** of the calculator task. LangGraph and CrewAI are not repeated here; see [12_langgraph_hands_on](12_langgraph_hands_on.ipynb) and [13_crewai_and_multi_agent_frameworks_hands_on](13_crewai_and_multi_agent_frameworks_hands_on.ipynb).

### 3.1 The Microsoft family: AutoGen, AG2, Semantic Kernel, and Microsoft Agent Framework

This family has the most confusing history, so here it is in one table (as of late 2026):

| Name | What happened | Status (late 2026) |
|---|---|---|
| **AutoGen 0.2** (2023) | the original Microsoft Research multi-agent "conversation" framework | old API; lives on as AG2 |
| **AG2** (`ag2`) | **community fork** of AutoGen 0.2 by some of its original creators (late 2024), separate GitHub org `ag2ai` | actively developed, independent of Microsoft, heading to a 1.0 release |
| **AutoGen 0.4+** (`autogen-agentchat`, `autogen-core`, `autogen-ext`) | Microsoft's full rewrite: event-driven core + a high-level "AgentChat" API | **maintenance mode / archived**; Microsoft points new users to Agent Framework |
| **Semantic Kernel** (`semantic-kernel`) | Microsoft's enterprise SDK (Python, C#, Java) for plugins, planners and agents | still maintained, but new agent work is meant to go to Agent Framework |
| **Microsoft Agent Framework** (`agent-framework`) | the **merger** of AutoGen's multi-agent ideas and Semantic Kernel's enterprise features, for Python and .NET | **the recommended path**; 1.0 GA in spring 2026 |

**Analogy.** AutoGen and Semantic Kernel were two teams in the same company building similar products. Microsoft merged them into one product (Agent Framework). Meanwhile part of the original AutoGen team left and kept the old design alive under a new name (AG2).

#### AutoGen AgentChat (0.4+ API)

**What it is.** Agents (`AssistantAgent`) that chat with each other in **teams** (`RoundRobinGroupChat`, `SelectorGroupChat`, `Swarm`, `MagenticOneGroupChat`), with **termination conditions** like "stop when someone says APPROVE".

**When to use / not.** Use it if you maintain existing AutoGen 0.4 code or want to study its clean design. For new projects, prefer Microsoft Agent Framework (same ideas, actively developed) or AG2 (if you liked the 0.2 style).

**Strengths:** clear team abstractions, good research pedigree (Magentic-One), async and event-driven. **Weaknesses:** maintenance mode; three generations of API in blog posts confuse beginners.

The demo uses the official test double **`ReplayChatCompletionClient`**: it replays a list of pre-written model replies.

In [4]:
async def autogen_calculator():
    from autogen_agentchat.agents import AssistantAgent
    from autogen_ext.models.replay import ReplayChatCompletionClient
    from autogen_core import FunctionCall
    from autogen_core.models import CreateResult, RequestUsage

    # Scripted model: reply 1 = a tool call, reply 2 = plain text.
    # (Replay clients cannot read the tool result, so reply 2 is fixed text; the event log below shows the real tool output.)
    model = ReplayChatCompletionClient(
        [CreateResult(finish_reason="function_calls",
                      content=[FunctionCall(id="call_1", name="multiply", arguments='{"a": 17, "b": 23}')],
                      usage=RequestUsage(prompt_tokens=0, completion_tokens=0), cached=False),
         "That is 391 pencils."],
        model_info={"vision": False, "function_calling": True, "json_output": False,
                    "family": "unknown", "structured_output": False})

    agent = AssistantAgent("calculator", model_client=model, tools=[multiply],   # a plain Python function becomes a tool
                           system_message="Use tools for maths.", reflect_on_tool_use=True)
    result = await agent.run(task=QUESTION)
    for msg in result.messages:                       # the event log of the run
        print(f"  {type(msg).__name__:24s} from {msg.source:10s}: {msg.content}")
    return result.messages[-1].content

await run_demo("AutoGen AgentChat", "calculator", autogen_calculator)

  TextMessage              from user      : A school orders 17 boxes with 23 pencils in each. How many pencils is that?
  ToolCallRequestEvent     from calculator: [FunctionCall(id='call_1', arguments='{"a": 17, "b": 23}', name='multiply')]
  ToolCallExecutionEvent   from calculator: [FunctionExecutionResult(content='391.0', name='multiply', call_id='call_1', is_error=False)]
  TextMessage              from calculator: That is 391 pencils.
[AutoGen AgentChat] final answer -> That is 391 pencils.


**What just happened?** The run produced four events: the user's task, a `ToolCallRequestEvent` (the model asked for `multiply`), a `ToolCallExecutionEvent` (AutoGen ran our Python function and got `391.0`), and the final `TextMessage`. `reflect_on_tool_use=True` means "after the tool runs, call the model once more to write a sentence".

#### Microsoft Agent Framework (MAF)

**What it is.** Microsoft's unified agent SDK for Python and .NET. Two layers: **agents** (`Agent` = a chat client + instructions + tools) and **workflows** (typed graphs of "executors", with checkpoints and human-in-the-loop). Ready-made **orchestrations**: sequential, concurrent, group chat, handoff and Magentic (a manager agent that plans for a team). Built-in OpenTelemetry tracing, middleware, MCP and A2A support, and connectors for Azure, OpenAI, Anthropic, Ollama and others.

**Analogy.** Semantic Kernel's "enterprise toolbox" bolted onto AutoGen's "team of agents".

**When to use / not.** A strong default if you are on Azure / Microsoft Foundry or in a .NET shop, or you are migrating AutoGen / Semantic Kernel code. Less natural if you want the smallest possible dependency or you are all-in on another cloud.

**Strengths:** one SDK for Python and .NET, production features (telemetry, middleware, checkpoints), official migration guides. **Weaknesses:** young as a unified product, large surface area, many optional packages.

The demo plugs in a tiny **custom chat client** (subclass of the framework's `BaseChatClient`). This is the same extension point Microsoft uses for its own model connectors.

In [5]:
async def maf_calculator():
    from agent_framework import Agent, BaseChatClient, ChatResponse, Message, Content, FunctionInvocationLayer, tool

    @tool
    def multiply_tool(a: float, b: float) -> float:
        '''Multiply two numbers.'''
        return a * b

    class ScriptedChatClient(FunctionInvocationLayer, BaseChatClient):
        '''Stand-in model. FunctionInvocationLayer gives it the "run tools automatically" behaviour.'''
        async def _reply(self, messages):
            for m in messages:                                 # did a tool already run?
                for c in m.contents:
                    if c.type == "function_result":
                        return ChatResponse(messages=[Message(role="assistant", contents=[f"That is {c.result} pencils."])])
            call = Content.from_function_call(call_id="call_1", name="multiply_tool", arguments={"a": 17, "b": 23})
            return ChatResponse(messages=[Message(role="assistant", contents=[call])])
        def _inner_get_response(self, *, messages, stream, options, **kwargs):
            return self._reply(messages)

    agent = Agent(client=ScriptedChatClient(), name="calculator",
                  instructions="Use tools for maths.", tools=[multiply_tool])
    response = await agent.run(QUESTION)
    for m in response.messages:
        print(f"  {m.role:10s} {[c.type for c in m.contents]}")
    return response.text

await run_demo("Microsoft Agent Framework", "calculator", maf_calculator)

  assistant  ['function_call']
  tool       ['function_result']
  assistant  ['text']
[Microsoft Agent Framework] final answer -> That is 391.0 pencils.


#### Semantic Kernel and AG2 (code only)

**Semantic Kernel** organises tools as **plugins** (classes whose methods are marked `@kernel_function`) attached to a **kernel**. Its agent classes (`ChatCompletionAgent`, and Azure / OpenAI assistant agents) now have a migration path to Agent Framework. **When to use:** existing SK code bases, especially in C#. **Not for:** new agent projects (start on Agent Framework).

```python
# pip install semantic-kernel   (needs a model key at run time)
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function

class MathPlugin:
    @kernel_function(description="Multiply two numbers.")
    def multiply(self, a: float, b: float) -> float:
        return a * b

agent = ChatCompletionAgent(service=OpenAIChatCompletion(), name="calculator",
                            instructions="Use tools for maths.", plugins=[MathPlugin()])
response = await agent.get_response(messages="What is 17 * 23?")
```

**AG2** keeps the AutoGen 0.2 style (`ConversableAgent`, `initiate_chat`, group chats) and adds newer pieces (swarm-style handoffs, realtime agents, MCP/A2A work). **When to use:** you like the classic AutoGen conversation style and want an independent, community-run project. **Not for:** teams that want a single big vendor behind their framework.

```python
# pip install ag2[openai]   (needs a model key at run time)
from autogen import ConversableAgent, LLMConfig
llm_config = LLMConfig(api_type="openai", model="<your-model>")
assistant = ConversableAgent("assistant", llm_config=llm_config, system_message="Use tools for maths.")
user = ConversableAgent("user", human_input_mode="NEVER")
assistant.register_for_llm(description="Multiply two numbers.")(multiply)
user.register_for_execution()(multiply)
user.initiate_chat(assistant, message="What is 17 * 23?", max_turns=3)
```
(AG2 is moving quickly toward 1.0; class names above are the long-standing ones. Check the AG2 docs for the current form.)

### 3.2 OpenAI Agents SDK

**What it is.** OpenAI's small, production-oriented agent SDK (Python and TypeScript), the successor to the experimental **Swarm** library. Core primitives:

| Primitive | Plain meaning |
|---|---|
| `Agent` | a model + instructions + tools (+ optional handoffs and guardrails) |
| `function_tool` | turns a Python function into a tool (schema from type hints and docstring) |
| **Handoff** | the model "calls" `transfer_to_<agent>` and the other agent takes over the conversation |
| **Guardrail** | input/output checks that run alongside the agent and can stop the run (a "tripwire") |
| **Session** | automatic conversation memory between runs (e.g. `SQLiteSession`) |
| **Tracing** | every run is recorded as a trace (viewable in the OpenAI dashboard or any OpenTelemetry backend) |
| `Runner` | runs the loop: `Runner.run(agent, input)` |

In 2026 the SDK also gained a **harness for long tasks** with **sandbox agents** (code runs in an isolated container from providers such as E2B, Modal or Daytona), plus MCP support and a realtime/voice layer.

**Analogy.** A thin "agent kit" from the model vendor: few parts, each well documented, close to the model API.

**When to use / not.** Great default when you mainly use OpenAI models and want handoffs, guardrails and tracing with little code. It is **provider-agnostic in principle** (custom `Model` classes, LiteLLM), but the best experience is with OpenAI. If you need explicit graphs with checkpoints, LangGraph or a workflow framework fits better.

**Strengths:** tiny API, handoffs and guardrails built in, tracing on by default, Python and JS versions. **Weaknesses:** less control over complex flows than graph frameworks; tracing defaults upload to OpenAI unless you change them.

The SDK ships an official test double, **`agents.testing.ScriptedModel`**, which we use below.

In [6]:
async def openai_agents_calculator():
    from agents import Agent, Runner, function_tool, set_tracing_disabled
    from agents.testing import ScriptedModel, assistant_message, function_call
    set_tracing_disabled(True)                        # keep everything local

    @function_tool
    def multiply_tool(a: float, b: float) -> float:
        '''Multiply two numbers.'''
        return a * b

    model = ScriptedModel([                           # one list per model call
        [function_call("multiply_tool", {"a": 17, "b": 23}, call_id="call_1")],
        [assistant_message("That is 391 pencils.")],
    ])
    agent = Agent(name="Calculator", instructions="Use tools for maths.", tools=[multiply_tool], model=model)
    result = await Runner.run(agent, QUESTION)
    for item in result.new_items:
        print(f"  {type(item).__name__:20s} {getattr(item, 'output', '')}")
    print(f"  model was called {len(model.calls)} times")
    return result.final_output

await run_demo("OpenAI Agents SDK", "calculator", openai_agents_calculator)

  ToolCallItem         
  ToolCallOutputItem   391.0
  MessageOutputItem    
  model was called 2 times
[OpenAI Agents SDK] final answer -> That is 391 pencils.


**Guardrails and sessions in 20 lines.** A **guardrail** below blocks homework requests before the agent answers. A **session** stores the conversation in SQLite (in memory here), so the second run "remembers" the user's name without us passing history by hand.

In [7]:
async def openai_agents_guardrail_and_session():
    from agents import (Agent, Runner, input_guardrail, GuardrailFunctionOutput,
                        InputGuardrailTripwireTriggered, SQLiteSession, set_tracing_disabled)
    from agents.testing import ScriptedModel, assistant_message
    set_tracing_disabled(True)

    @input_guardrail
    async def no_homework(ctx, agent, user_input):
        is_homework = "homework" in str(user_input).lower()     # a real guardrail might call a small classifier model
        return GuardrailFunctionOutput(output_info={"homework": is_homework}, tripwire_triggered=is_homework)

    guarded = Agent(name="Tutor", instructions="Explain ideas.", input_guardrails=[no_homework],
                    model=ScriptedModel([[assistant_message("Here is a hint...")]]))
    try:
        await Runner.run(guarded, "Please do my homework for me")
        print("  guardrail: not triggered")
    except InputGuardrailTripwireTriggered:
        print("  guardrail: request blocked before the agent answered")

    session = SQLiteSession("demo-user")                         # in-memory SQLite by default
    model = ScriptedModel([[assistant_message("Nice to meet you, Ana.")], [assistant_message("Your name is Ana.")]])
    friendly = Agent(name="Friendly", instructions="Be kind.", model=model)
    await Runner.run(friendly, "Hi, my name is Ana.", session=session)
    second = await Runner.run(friendly, "What is my name?", session=session)
    print(f"  items the model saw on the 2nd run: {len(model.calls[1].input)} (history was added by the session)")
    return second.final_output

await run_demo("OpenAI Agents SDK", "guardrail + session", openai_agents_guardrail_and_session)

  guardrail: request blocked before the agent answered
  items the model saw on the 2nd run: 3 (history was added by the session)
[OpenAI Agents SDK] final answer -> Your name is Ana.


**OpenAI Swarm (historical).** `swarm` (2024) was an **educational** library that introduced the "agents + handoffs as function calls" idea in a few hundred lines. OpenAI marked it experimental and **replaced it with the Agents SDK**. Read its source to learn; do not build on it.

```python
# historical - do not use for new projects
from swarm import Swarm, Agent
def transfer_to_writer(): return writer
researcher = Agent(name="Researcher", functions=[transfer_to_writer])
writer = Agent(name="Writer", instructions="Write a short paragraph.")
Swarm().run(agent=researcher, messages=[{"role": "user", "content": "Write about agents"}])
```

### 3.3 Google Agent Development Kit (ADK)

**What it is.** Google's open-source agent framework (Python, Java, Go, and TypeScript). Agents (`LlmAgent`) plus "workflow agents" (`SequentialAgent`, `ParallelAgent`, `LoopAgent`), a `Runner` with pluggable **session**, **memory** and **artifact** services, a local dev web UI (`adk web`), an evaluation tool, and first-class **A2A** support. **ADK 2.0** (2026) added a graph-based **workflow runtime** with human-in-the-loop. Works best with Gemini, but other models plug in through `BaseLlm` subclasses or LiteLLM.

**Analogy.** A full "agent workshop" from Google: build, test in a local UI, evaluate, then deploy to Google Cloud.

**When to use / not.** Good fit for Google Cloud / Gemini users, multi-agent hierarchies, and A2A. If you never touch Google Cloud and want something tiny, it can feel heavy.

**Strengths:** complete toolchain (dev UI, eval, deploy), multi-language, A2A native. **Weaknesses:** large dependency tree; fast-moving 1.x -> 2.x changes.

**Naming note (late 2026):** Google rebranded **Vertex AI** as the **Gemini Enterprise Agent Platform** (April 2026); "Agent Engine", the managed place to deploy ADK agents, now sits under that platform (see section 5).

The demo subclasses ADK's `BaseLlm` with a scripted model.

In [8]:
async def adk_calculator():
    from google.adk.agents import LlmAgent
    from google.adk.models.base_llm import BaseLlm
    from google.adk.models.llm_response import LlmResponse
    from google.adk.runners import InMemoryRunner
    from google.genai import types

    class ScriptedLlm(BaseLlm):
        '''Stand-in for Gemini: same interface, scripted replies.'''
        model: str = "scripted"
        async def generate_content_async(self, llm_request, stream=False):
            last = llm_request.contents[-1]
            results = [p.function_response for p in last.parts if p.function_response]
            if results:   # the tool already ran -> answer with its result
                parts = [types.Part(text=f"That is {results[0].response['result']} pencils.")]
            else:
                parts = [types.Part(function_call=types.FunctionCall(name="multiply", args={"a": 17, "b": 23}))]
            yield LlmResponse(content=types.Content(role="model", parts=parts))

    agent = LlmAgent(name="calculator", model=ScriptedLlm(), instruction="Use tools for maths.", tools=[multiply])
    runner = InMemoryRunner(agent=agent, app_name="demo")           # in-memory session service
    session = await runner.session_service.create_session(app_name="demo", user_id="u1")
    message = types.Content(role="user", parts=[types.Part(text=QUESTION)])
    final = None
    async for event in runner.run_async(user_id="u1", session_id=session.id, new_message=message):
        for p in (event.content.parts if event.content else []):
            if p.function_call:     print(f"  event: tool call   {p.function_call.name}{dict(p.function_call.args)}")
            elif p.function_response: print(f"  event: tool result {p.function_response.response}")
            elif p.text:            print(f"  event: text        {p.text}"); final = p.text
    return final

await run_demo("Google ADK", "calculator", adk_calculator)

  event: tool call   multiply{'a': 17, 'b': 23}
  event: tool result {'result': 391}
  event: text        That is 391 pencils.
[Google ADK] final answer -> That is 391 pencils.


**What just happened?** ADK runs are a stream of **events** (tool call, tool result, text). The `InMemoryRunner` stored them in a session; in production you would swap in a database or cloud session service without changing the agent.

### 3.4 Claude Agent SDK

**What it is.** Anthropic's **agent harness as a library**: the same agent loop that powers Claude Code, packaged for Python (`claude-agent-sdk`) and TypeScript (`@anthropic-ai/claude-agent-sdk`). It ships built-in tools (read/write/edit files, bash, grep, web search/fetch), **context management**, **permissions**, **hooks** (code that runs before/after tool calls), **subagents**, sessions, and MCP. You give it a prompt and options; it drives the whole loop.

**Analogy.** Not a box of Lego bricks, but a ready-made robot you can configure and give extra tools to.

**How it differs from the plain Claude API.** The Claude API's tool-use loop (or its SDK "tool runner") only runs tools **you** define. The Agent SDK is the full harness with file and shell tools built in. Neither one hosts your agent; you run it on your own machine or server.

**When to use / not.** Great for coding agents, file/ops automation, research agents that work in a folder, and long multi-step tasks. Overkill for a simple chatbot with two tools (use the API directly or a lighter framework). Needs Claude credentials, so it is shown as code only.

**Strengths:** battle-tested harness, strong built-in tools, hooks and permissions for safety. **Weaknesses:** Claude-specific; built-in shell/file tools need careful permission settings.

```python
# pip install claude-agent-sdk   (needs Claude credentials at run time)
from typing import Any
from claude_agent_sdk import query, ClaudeAgentOptions, AgentDefinition, tool, create_sdk_mcp_server

@tool("multiply", "Multiply two numbers", {"a": float, "b": float})
async def multiply_tool(args: dict[str, Any]) -> dict[str, Any]:
    return {"content": [{"type": "text", "text": str(args["a"] * args["b"])}]}

calculator = create_sdk_mcp_server(name="calc", version="1.0.0", tools=[multiply_tool])  # in-process MCP server

options = ClaudeAgentOptions(
    system_prompt="Use tools for maths.",
    mcp_servers={"calc": calculator},
    allowed_tools=["mcp__calc__multiply"],          # tools named mcp__<server>__<tool>
    agents={"writer": AgentDefinition(description="Writes short paragraphs",
                                      prompt="You turn notes into one clear paragraph.")},  # a subagent
)
async for message in query(prompt="How many pencils are in 17 boxes of 23?", options=options):
    print(message)
```

### 3.5 PydanticAI

**What it is.** An agent framework from the team behind Pydantic (the data-validation library). Its big idea is **type safety**: tools, dependencies and outputs are typed Python objects, and the output is validated (if the model returns bad data, PydanticAI can ask it to retry). It is model-agnostic, has **dependency injection** (pass a database connection to tools cleanly), MCP support, durable execution integrations (e.g. Temporal), graph support (`pydantic-graph`), and tracing via Logfire / OpenTelemetry.

**Analogy.** FastAPI for agents: type hints in, validated objects out.

**When to use / not.** Excellent when your agent must return **structured data** that other code uses (a form, a JSON record), and when you care about tests. For big "team of agents" setups you write more orchestration yourself.

**Strengths:** clean typed API, superb testing tools (`TestModel`, `FunctionModel`), works with many providers. **Weaknesses:** multi-agent patterns are "just Python", not a built-in team abstraction; version numbers move fast.

PydanticAI ships two test models: **`TestModel`** (auto-generates valid fake data) and **`FunctionModel`** (you write the model as a Python function). We use both.

In [9]:
async def pydantic_ai_calculator():
    from pydantic_ai import Agent
    from pydantic_ai.models.function import FunctionModel, AgentInfo
    from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, ToolReturnPart

    agent = Agent(instructions="Use tools for maths.")

    @agent.tool_plain                      # register a tool that needs no context
    def multiply_tool(a: float, b: float) -> float:
        '''Multiply two numbers.'''
        return a * b

    def scripted_model(messages, info: AgentInfo) -> ModelResponse:
        for part in messages[-1].parts:    # did the last request carry a tool result?
            if isinstance(part, ToolReturnPart):
                return ModelResponse(parts=[TextPart(f"That is {part.content} pencils.")])
        return ModelResponse(parts=[ToolCallPart("multiply_tool", {"a": 17, "b": 23})])

    result = await agent.run(QUESTION, model=FunctionModel(scripted_model))
    for m in result.all_messages():
        print(f"  {type(m).__name__:14s} {[type(p).__name__ for p in m.parts]}")
    return result.output

await run_demo("PydanticAI", "calculator", pydantic_ai_calculator)

  ModelRequest   ['UserPromptPart']
  ModelResponse  ['ToolCallPart']
  ModelRequest   ['ToolReturnPart']
  ModelResponse  ['TextPart']
[PydanticAI] final answer -> That is 391.0 pencils.


In [10]:
async def pydantic_ai_structured_output():
    from pydantic import BaseModel, Field
    from pydantic_ai import Agent
    from pydantic_ai.models.test import TestModel

    class PencilOrder(BaseModel):                       # the shape we want back, as a typed object
        boxes: int = Field(ge=1)
        per_box: int = Field(ge=1)
        total: int

    agent = Agent(output_type=PencilOrder, instructions="Extract the order.")
    # TestModel invents data that fits the schema - perfect for testing the plumbing without an LLM
    result = await agent.run(QUESTION, model=TestModel(custom_output_args={"boxes": 17, "per_box": 23, "total": 391}))
    order = result.output
    print(f"  type: {type(order).__name__}, fields: {order.model_dump()}")
    return f"PencilOrder(total={order.total})"

await run_demo("PydanticAI", "structured output", pydantic_ai_structured_output)

  type: PencilOrder, fields: {'boxes': 17, 'per_box': 23, 'total': 391}
[PydanticAI] final answer -> PencilOrder(total=391)


**What just happened?** In the first run, PydanticAI produced the classic 4-message trace (request, tool call, tool return, text). In the second, the agent's output is not a string but a validated `PencilOrder` object; if the model had returned `boxes = 0`, validation would fail and PydanticAI would ask the model to try again.

### 3.6 Hugging Face smolagents

**What it is.** A deliberately small library (the core is about a thousand lines) from Hugging Face. Its signature idea is the **CodeAgent**: instead of emitting JSON tool calls, the model writes a short **Python snippet** that calls tools, and smolagents executes it. It also has a classic **ToolCallingAgent**. Works with any model (Hugging Face Inference, local `transformers`, LiteLLM, OpenAI, ...), can share tools on the Hugging Face Hub, and supports MCP.

**CodeAgent vs ToolCallingAgent:**

| | CodeAgent | ToolCallingAgent |
|---|---|---|
| Action format | Python code: `result = multiply(17, 23)` | JSON: `{"name": "multiply", "arguments": {...}}` |
| Can do loops / combine tools in one step | yes | no, one call per step |
| Steps needed for complex tasks | usually fewer | usually more |
| Safety | must run code in a **sandbox** (local restricted interpreter, or E2B / Docker / Modal etc.) | safer by default |
| Best for | data work, multi-step logic, research | simple, predictable tool use |

**Analogy.** Instead of filling a form for each action (JSON), the agent writes a tiny script.

**When to use / not.** Great for learning, research, and with open-weight models. For strict enterprise flows, the code-execution risk needs a real sandbox.

**Strengths:** tiny and readable, model-agnostic, code actions are powerful. **Weaknesses:** fewer production features (durable state, HITL); running model-written code must be sandboxed.

The demo subclasses smolagents' `Model` with a scripted model. The CodeAgent's "code" is run by smolagents' **restricted local Python interpreter**.

In [11]:
def smolagents_code_agent():
    from smolagents import CodeAgent, Model, tool
    from smolagents.models import ChatMessage, MessageRole

    @tool
    def multiply_tool(a: float, b: float) -> float:
        '''Multiply two numbers.

        Args:
            a: the first number
            b: the second number
        '''
        return a * b

    class ScriptedCodeModel(Model):
        '''Stand-in model that "writes" one Python action.'''
        def generate(self, messages, stop_sequences=None, response_format=None, tools_to_call_from=None, **kw):
            action = ("Thought: multiply boxes by pencils per box.\n<code>\n"
                      "total = multiply_tool(17, 23)\nfinal_answer(f'That is {int(total)} pencils.')\n</code>")
            return ChatMessage(role=MessageRole.ASSISTANT, content=action)

    agent = CodeAgent(tools=[multiply_tool], model=ScriptedCodeModel(model_id="scripted"), verbosity_level=0)
    answer = agent.run(QUESTION)
    print("  code the model wrote and smolagents executed:")
    print("   ", agent.memory.steps[-1].code_action.replace("\n", "\n    "))
    return answer

await run_demo("smolagents", "calculator (CodeAgent)", smolagents_code_agent)

  code the model wrote and smolagents executed:
    total = multiply_tool(17, 23)
    final_answer(f'That is {int(total)} pencils.')
[smolagents] final answer -> That is 391 pencils.


In [12]:
def smolagents_tool_calling_agent():
    from smolagents import ToolCallingAgent, Model, tool
    from smolagents.models import ChatMessage, MessageRole, ChatMessageToolCall, ChatMessageToolCallFunction

    @tool
    def multiply_tool(a: float, b: float) -> float:
        '''Multiply two numbers.

        Args:
            a: the first number
            b: the second number
        '''
        return a * b

    class ScriptedToolModel(Model):
        '''Stand-in model: step 1 calls multiply_tool, step 2 calls final_answer using the observation.'''
        def generate(self, messages, stop_sequences=None, response_format=None, tools_to_call_from=None, **kw):
            text = json.dumps([m.dict() if hasattr(m, "dict") else str(m) for m in messages], default=str)
            if "391" not in text:   # tool not run yet
                fn = ChatMessageToolCallFunction(name="multiply_tool", arguments={"a": 17, "b": 23})
            else:
                fn = ChatMessageToolCallFunction(name="final_answer", arguments={"answer": "That is 391 pencils."})
            return ChatMessage(role=MessageRole.ASSISTANT, content="",
                               tool_calls=[ChatMessageToolCall(id="c1", type="function", function=fn)])

    agent = ToolCallingAgent(tools=[multiply_tool], model=ScriptedToolModel(model_id="scripted"), verbosity_level=0)
    answer = agent.run(QUESTION)
    print(f"  steps recorded in memory: {[type(s).__name__ for s in agent.memory.steps]}")
    return answer

await run_demo("smolagents", "calculator (ToolCallingAgent)", smolagents_tool_calling_agent)

  steps recorded in memory: ['TaskStep', 'ActionStep', 'ActionStep']
[smolagents] final answer -> That is 391 pencils.


### 3.7 LlamaIndex: Workflows and AgentWorkflow

**What it is.** LlamaIndex began as the "connect LLMs to your data" (RAG) library. Its agent layer has two parts:
- **Workflows**: an **event-driven** way to write any multi-step program. Each `@step` function receives an event and returns the next event; the workflow wires them by type. No LLM is required, which makes them easy to test. (Workflows now also ship as a standalone package, `llama-index-workflows`.)
- **Agents on top**: `FunctionAgent`, `ReActAgent`, `CodeActAgent`, and **`AgentWorkflow`** for several agents with **handoffs**.

**Analogy.** A relay race: each runner (step) waits for a specific baton (event type), runs, and hands on a new baton.

**When to use / not.** Best when your agent is mostly about **documents and retrieval** (LlamaIndex's data connectors, indexes and LlamaParse are strong), or you like event-driven code. If you need no RAG at all, lighter options exist.

**Strengths:** first-class RAG tooling, clean event model, good async story. **Weaknesses:** big ecosystem with many packages; API names changed several times (older "AgentRunner"/"OpenAIAgent" tutorials are outdated).

First, a pure **Workflow** with no model at all:

In [13]:
async def llamaindex_workflow_no_llm():
    from llama_index.core.workflow import Workflow, StartEvent, StopEvent, Event, step

    class NumbersParsed(Event):            # a custom event type = the "baton"
        boxes: int
        per_box: int

    class PencilWorkflow(Workflow):
        @step
        async def parse(self, ev: StartEvent) -> NumbersParsed:
            nums = [int(w) for w in ev.question.replace("?", "").split() if w.isdigit()]
            print(f"  step parse    -> found numbers {nums}")
            return NumbersParsed(boxes=nums[0], per_box=nums[1])

        @step
        async def compute(self, ev: NumbersParsed) -> StopEvent:
            total = int(multiply(ev.boxes, ev.per_box))
            print(f"  step compute  -> {ev.boxes} x {ev.per_box} = {total}")
            return StopEvent(result=f"That is {total} pencils.")

    return await PencilWorkflow(timeout=10).run(question=QUESTION)

await run_demo("LlamaIndex", "workflow without an LLM", llamaindex_workflow_no_llm)

  step parse    -> found numbers [17, 23]
  step compute  -> 17 x 23 = 391
[LlamaIndex] final answer -> That is 391 pencils.


Now the same calculator as an LLM agent. LlamaIndex ships **`MockFunctionCallingLLM`**; we give it a scripted `response_generator`.

In [14]:
async def llamaindex_function_agent():
    from llama_index.core.llms.mock import MockFunctionCallingLLM
    from llama_index.core.llms import ChatMessage, MessageRole
    from llama_index.core.base.llms.types import ToolCallBlock
    from llama_index.core.agent.workflow import FunctionAgent

    def script(messages, **kwargs):
        tool_msgs = [m for m in messages if m.role == MessageRole.TOOL]
        if tool_msgs:
            return ChatMessage(role="assistant", content=f"That is {tool_msgs[-1].content} pencils.")
        return ChatMessage(role="assistant", blocks=[
            ToolCallBlock(tool_call_id="call_1", tool_name="multiply", tool_kwargs={"a": 17, "b": 23})])

    agent = FunctionAgent(tools=[multiply], llm=MockFunctionCallingLLM(response_generator=script),
                          system_prompt="Use tools for maths.")
    return str(await agent.run(QUESTION))

await run_demo("LlamaIndex", "calculator", llamaindex_function_agent)

[LlamaIndex] final answer -> That is 391 pencils.


### 3.8 Haystack (deepset)

**What it is.** A framework for building **pipelines**: you add **components** (retriever, prompt builder, generator, router, ...) and **connect** their inputs and outputs into a graph. Haystack has an `Agent` component (a chat generator + tools + an exit condition) that can sit inside a pipeline. Strong at search and RAG, with many document-store integrations, serialisable pipelines (YAML), and an enterprise platform from deepset.

**Analogy.** Plumbing: you choose pipes and valves and connect them; water (data) flows along fixed routes.

**When to use / not.** Production RAG and search systems where you want explicit, testable, serialisable pipelines; agents as one part of a bigger system. If you want free-form multi-agent chat, other frameworks are more natural.

**Strengths:** explicit pipelines, mature RAG components, good for teams. **Weaknesses:** more boilerplate for simple agents; agent features are newer than its pipeline core.

First a pipeline with **no LLM at all** (keyword retrieval + prompt building), then the `Agent` component with a scripted chat generator.

In [15]:
def haystack_pipeline_no_llm():
    from haystack import Pipeline, Document
    from haystack.document_stores.in_memory import InMemoryDocumentStore
    from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
    from haystack.components.builders import PromptBuilder

    store = InMemoryDocumentStore()
    store.write_documents([Document(content="PydanticAI is a type-safe Python agent framework."),
                           Document(content="Mastra is a TypeScript agent framework."),
                           Document(content="smolagents lets the model write Python code as its actions.")])
    pipe = Pipeline()
    pipe.add_component("retriever", InMemoryBM25Retriever(document_store=store, top_k=1))   # keyword search
    pipe.add_component("prompt", PromptBuilder(
        template="Question: {{query}}\nContext:\n{% for d in documents %}- {{d.content}}\n{% endfor %}",
        required_variables=["query", "documents"]))
    pipe.connect("retriever.documents", "prompt.documents")          # draw the arrow between components
    q = "Which framework is for TypeScript?"
    out = pipe.run({"retriever": {"query": q}, "prompt": {"query": q}})
    print("  prompt that would be sent to an LLM:\n    " + out["prompt"]["prompt"].strip().replace("\n", "\n    "))
    return out["prompt"]["prompt"].strip().splitlines()[-1]

await run_demo("Haystack", "pipeline without an LLM", haystack_pipeline_no_llm)

  prompt that would be sent to an LLM:
    Question: Which framework is for TypeScript?
    Context:
    - Mastra is a TypeScript agent framework.
[Haystack] final answer -> - Mastra is a TypeScript agent framework.


In [16]:
def haystack_agent():
    from typing import Any, List, Optional
    from haystack import component
    from haystack.components.agents import Agent
    from haystack.dataclasses import ChatMessage, ToolCall
    from haystack.tools import create_tool_from_function

    @component
    class ScriptedChatGenerator:
        '''Stand-in for a chat model component (same input/output sockets as the real ones).'''
        @component.output_types(replies=List[ChatMessage])
        def run(self, messages: List[ChatMessage], tools: Optional[Any] = None, **kwargs):
            results = [r for m in messages for r in m.tool_call_results]
            if results:
                return {"replies": [ChatMessage.from_assistant(f"That is {results[-1].result} pencils.")]}
            call = ToolCall(tool_name="multiply", arguments={"a": 17, "b": 23}, id="call_1")
            return {"replies": [ChatMessage.from_assistant(tool_calls=[call])]}

    agent = Agent(chat_generator=ScriptedChatGenerator(), tools=[create_tool_from_function(multiply)],
                  system_prompt="Use tools for maths.")
    out = agent.run(messages=[ChatMessage.from_user(QUESTION)])
    print(f"  roles in the run: {[m.role.value for m in out['messages']]}")
    return out["messages"][-1].text

await run_demo("Haystack", "calculator", haystack_agent)

  roles in the run: ['system', 'user', 'assistant', 'tool', 'assistant']
[Haystack] final answer -> That is 391 pencils.


### 3.9 Agno (formerly Phidata)

**What it is.** **Phidata was renamed Agno** (early 2025). A fast, batteries-included Python framework with three levels: **Agent** (model + tools + memory + knowledge), **Team** (several agents with a leader, in modes such as coordinate, route, broadcast and tasks) and **Workflow** (deterministic steps). It ships many tool integrations, storage and memory backends, and **AgentOS**, a FastAPI-based runtime/control plane to serve agents as APIs (and, since 2026, to host agents built with other frameworks too).

**Analogy.** An all-in-one kit: agent, memory, knowledge base and a server, all from one package.

**When to use / not.** Quick to get a useful agent with memory and a web API running. If you want a minimal library or very explicit graphs, look elsewhere.

**Strengths:** fast, lots of integrations, memory and storage built in, AgentOS for serving. **Weaknesses:** big surface; **telemetry is on by default** (set `AGNO_TELEMETRY=false` or `telemetry=False`, as we did in Setup); frequent major versions.

The demo subclasses Agno's `Model` base class.

In [17]:
async def agno_calculator():
    from dataclasses import dataclass
    from agno.agent import Agent
    from agno.models.base import Model
    from agno.models.response import ModelResponse

    @dataclass
    class ScriptedModel(Model):
        '''Stand-in model. Agno calls invoke(); we return a tool call first, then text.'''
        id: str = "scripted"
        name: str = "ScriptedModel"
        provider: str = "local"
        def invoke(self, messages, assistant_message=None, **kw):
            tool_msgs = [m for m in messages if m.role == "tool"]
            if tool_msgs:
                return ModelResponse(role="assistant", content=f"That is {tool_msgs[-1].content} pencils.")
            call = {"id": "call_1", "type": "function",
                    "function": {"name": "multiply", "arguments": json.dumps({"a": 17, "b": 23})}}
            return ModelResponse(role="assistant", content="", tool_calls=[call])
        async def ainvoke(self, *a, **kw): return self.invoke(*a, **kw)
        def invoke_stream(self, *a, **kw): yield self.invoke(*a, **kw)
        async def ainvoke_stream(self, *a, **kw): yield self.invoke(*a, **kw)
        def _parse_provider_response(self, response, **kw): return response
        def _parse_provider_response_delta(self, response): return response

    agent = Agent(model=ScriptedModel(), tools=[multiply], instructions="Use tools for maths.", telemetry=False)
    run = await agent.arun(QUESTION)
    print(f"  roles in the run: {[m.role for m in run.messages]}")
    return run.content

await run_demo("Agno", "calculator", agno_calculator)

  roles in the run: ['system', 'user', 'assistant', 'tool', 'assistant']
[Agno] final answer -> That is 391.0 pencils.


### 3.10 AWS Strands Agents (and Bedrock AgentCore)

**What it is.** **Strands Agents** is AWS's open-source, **model-driven** SDK (Python and TypeScript): you give an agent a model, a prompt and tools, and let the model plan. Multi-agent primitives (since 1.0): **agents-as-tools**, **swarm**, **graph** and **workflow**, plus A2A and MCP. Works with Bedrock, Anthropic, OpenAI, Ollama, LiteLLM and others. **Amazon Bedrock AgentCore** is the separate **managed runtime** (hosting, memory, identity, gateway for tools, observability, evaluations) that can run Strands agents or agents from other frameworks (section 5).

**Analogy.** Strands is the engine; AgentCore is the garage that runs and maintains it.

**When to use / not.** Natural choice on AWS, and a clean minimal SDK in general. If you need rich explicit graphs with time travel, LangGraph is more mature there.

**Strengths:** very small API (`Agent(tools=[...])("question")`), good AWS integration, open source. **Weaknesses:** younger ecosystem; best features assume AWS.

The demo subclasses Strands' `Model` and emits the same stream events Bedrock would.

In [18]:
async def strands_calculator():
    from strands import Agent, tool
    from strands.models import Model

    @tool
    def multiply_tool(a: float, b: float) -> float:
        '''Multiply two numbers.'''
        return a * b

    class ScriptedModel(Model):
        '''Stand-in model: yields Bedrock-style stream events.'''
        def __init__(self): self.config = {}
        def update_config(self, **cfg): self.config.update(cfg)
        def get_config(self): return self.config
        async def structured_output(self, output_model, prompt, system_prompt=None, **kw):
            raise NotImplementedError("not needed for this demo")
            yield
        async def stream(self, messages, tool_specs=None, system_prompt=None, **kw):
            result = next((b["toolResult"]["content"][0]["text"] for b in messages[-1]["content"] if "toolResult" in b), None)
            yield {"messageStart": {"role": "assistant"}}
            if result is None:
                yield {"contentBlockStart": {"start": {"toolUse": {"toolUseId": "t1", "name": "multiply_tool"}}}}
                yield {"contentBlockDelta": {"delta": {"toolUse": {"input": json.dumps({"a": 17, "b": 23})}}}}
                yield {"contentBlockStop": {}}
                yield {"messageStop": {"stopReason": "tool_use"}}
            else:
                yield {"contentBlockDelta": {"delta": {"text": f"That is {result} pencils."}}}
                yield {"contentBlockStop": {}}
                yield {"messageStop": {"stopReason": "end_turn"}}

    agent = Agent(model=ScriptedModel(), tools=[multiply_tool], system_prompt="Use tools for maths.",
                  callback_handler=None)              # None = do not print streaming text
    result = await agent.invoke_async(QUESTION)
    print(f"  conversation: {[(m['role'], list(m['content'][0].keys())[0]) for m in agent.messages]}")
    return str(result).strip()

await run_demo("Strands Agents", "calculator", strands_calculator)

  conversation: [('user', 'text'), ('assistant', 'toolUse'), ('user', 'toolResult'), ('assistant', 'text')]
[Strands Agents] final answer -> That is 391.0 pencils.


### 3.11 DSPy: programming, not prompting

**What it is.** DSPy (Stanford NLP) treats an LM pipeline as a **program**. You declare **signatures** (what goes in and out, e.g. `"question -> answer"`), compose **modules** (`Predict`, `ChainOfThought`, `ReAct` for tool-using agents), and then an **optimizer** (e.g. MIPROv2, GEPA) automatically improves the prompts and few-shot examples using a metric and some training examples.

**Analogy.** Other frameworks: you hand-write the recipe card (prompt). DSPy: you say "I want a cake, here is how I will taste it", and it rewrites the recipe card until the cake tastes best.

**When to use / not.** When quality matters and you can measure it (you have examples + a metric), and when you swap models often (re-optimise instead of re-writing prompts). Not ideal if you want hand-crafted prompts or complex multi-agent orchestration with HITL.

**Strengths:** systematic prompt optimisation, model portability, small composable modules. **Weaknesses:** a different mental model; optimisation costs many model calls; agent features are simpler than dedicated agent frameworks.

DSPy ships **`DummyLM`**, which returns pre-written field values. In DSPy 3.x, `ReAct` finishes by calling a built-in `submit` tool.

In [19]:
def dspy_calculator():
    import dspy
    from dspy.utils.dummies import DummyLM

    # 1) A plain signature: question -> answer
    dspy.configure(lm=DummyLM([{"answer": "391"}]))
    print("  Predict:", dspy.Predict("question -> answer")(question=QUESTION).answer)

    # 2) A ReAct agent with our tool. Each dict = the fields one model call "returns".
    dspy.configure(lm=DummyLM([
        {"next_thought": "Multiply boxes by pencils per box.",
         "tool_calls": {"tool_calls": [{"name": "multiply", "args": {"a": 17, "b": 23}}]}},
        {"next_thought": "I have the total.",
         "tool_calls": {"tool_calls": [{"name": "submit", "args": {"answer": "That is 391 pencils."}}]}},
    ]))
    agent = dspy.ReAct("question -> answer", tools=[multiply], max_iters=3)
    pred = agent(question=QUESTION)
    for turn in pred.history.messages:
        for call in turn["tool_calls"].tool_calls:
            res = turn["tool_calls"].tool_call_results.tool_call_results[0].value
            print(f"  ReAct turn: thought={turn['next_thought']!r:40s} tool={call.name}{call.args} -> {res}")
    return pred.answer

await run_demo("DSPy", "calculator", dspy_calculator)

  Predict: 391
  ReAct turn: thought='Multiply boxes by pencils per box.'     tool=multiply{'a': 17, 'b': 23} -> 391.0
  ReAct turn: thought='I have the total.'                      tool=submit{'answer': 'That is 391 pencils.'} -> {'answer': 'That is 391 pencils.'}
[DSPy] final answer -> That is 391 pencils.


**What just happened?** `ReAct` ran a real tool loop: our `multiply` executed (391.0 came from Python, not the script), then the model "submitted" the answer. With a real LM, you would next call an optimizer, e.g. `dspy.MIPROv2(metric=my_metric).compile(agent, trainset=examples)`, to improve the prompts automatically.

### 3.12 LangChain Deep Agents

**What it is.** `deepagents` is LangChain's **agent harness** on top of LangGraph: one call, `create_deep_agent(...)`, gives you an agent with a **planning tool** (a to-do list), a **virtual file system** for notes, **sub-agents** for focused sub-tasks, and a detailed system prompt, the pattern popularised by coding agents like Claude Code. The stack is: LangGraph (runtime) -> LangChain `create_agent` (simple harness) -> Deep Agents (opinionated long-task harness).

**When to use / not.** Long, multi-step research or coding tasks where you want LangGraph's durability without designing the graph yourself. For short tool calls, plain `create_agent` is enough. See [12_langgraph_hands_on](12_langgraph_hands_on.ipynb) for the runtime underneath.

```python
# pip install deepagents   (needs a model key at run time)
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="<provider:model-name>",
    tools=[multiply],
    system_prompt="You are a careful research assistant.",
    subagents=[{"name": "writer", "description": "Turns notes into a paragraph",
                "system_prompt": "Write one clear paragraph from the notes."}],
)
result = agent.invoke({"messages": [{"role": "user", "content": "Research agent frameworks, then write a summary"}]})
```

### 3.13 Letta (formerly MemGPT)

**What it is.** **MemGPT** (the 2023 paper) became the company and framework **Letta**. Letta agents are **stateful services**: they live on a Letta server, keep **memory blocks** (always-in-context facts such as "human" and "persona"), **archival memory** (a searchable long-term store), and edit their own memory with tools. You talk to an agent by ID, days apart, and it remembers. See [07_advanced_agent_frameworks](07_advanced_agent_frameworks.ipynb) for the memory-hierarchy idea.

**When to use / not.** Personal assistants, companions, support agents that must remember users over weeks. Not needed for one-shot tasks.

```python
# pip install letta-client   (needs a running Letta server or Letta Cloud)
from letta_client import Letta
client = Letta(base_url="http://localhost:8283")
agent = client.agents.create(model="<provider/model>", embedding="<provider/embedding-model>",
                             memory_blocks=[{"label": "human", "value": "Name: Ana. Likes short answers."},
                                            {"label": "persona", "value": "A friendly tutor."}])
reply = client.agents.messages.create(agent_id=agent.id, messages=[{"role": "user", "content": "Remember my exam is Friday."}])
```

### 3.14 Research-style multi-agent frameworks: CAMEL, MetaGPT, ChatDev

| Framework | Core idea | Use it for | Status (late 2026) |
|---|---|---|---|
| **CAMEL** (`camel-ai`) | **role-playing**: an "AI user" and an "AI assistant" agent instruct each other to solve a task; also "societies" (Workforce) of many agents, data generation, and research on agent scaling | research, synthetic data generation, studying agent societies | active open-source research community |
| **MetaGPT** (`metagpt`) | a **software company** of roles (product manager, architect, engineer) passing structured documents (SOPs) | studying structured multi-agent pipelines; generating small projects | the team's focus moved to the MGX product; open-source library still available |
| **ChatDev** | a virtual software company where roles chat in phases (design, coding, testing) | research and teaching | research project (OpenBMB) |

```python
# pip install camel-ai   (needs a model key at run time)
from camel.societies import RolePlaying
session = RolePlaying(assistant_role_name="Python programmer", user_role_name="Teacher",
                      task_prompt="Write a function that multiplies two numbers and test it.")
```
MetaGPT is covered in more depth in [07_advanced_agent_frameworks](07_advanced_agent_frameworks.ipynb); CrewAI (the most popular "crew of roles" framework) is in [13_crewai_and_multi_agent_frameworks_hands_on](13_crewai_and_multi_agent_frameworks_hands_on.ipynb).

### 3.15 Smaller but notable: BeeAI, Langroid, Atomic Agents

| Framework | What it is | Strengths | Weaknesses | Status (late 2026) |
|---|---|---|---|---|
| **BeeAI Framework** (`beeai-framework`) | IBM-started framework (Python + TypeScript) for production agents, with requirement-based agents and workflows; donated to the **Linux Foundation** (LF AI & Data) in 2025. Its platform became **Agent Stack**; its ACP protocol **merged into A2A** | open governance, two languages, A2A/MCP | smaller community | active, Linux Foundation project |
| **Langroid** (`langroid`) | multi-agent framework from CMU/UW-Madison researchers, built around agents exchanging **messages** and **tasks** (actor-style) | simple mental model, light dependencies, good docs | smaller community | active |
| **Atomic Agents** (`atomic-agents`) | "LEGO bricks": every agent and tool has a **typed input schema and output schema** (built on Instructor + Pydantic); you chain them explicitly | very predictable, easy to test, no magic | you write more orchestration yourself | active |

```python
# Atomic Agents style (sketch): typed in, typed out, chained by you
from atomic_agents import AtomicAgent, AgentConfig, BaseIOSchema
class Question(BaseIOSchema):
    '''A maths question.'''
    text: str
class Answer(BaseIOSchema):
    '''The final answer.'''
    value: int
agent = AtomicAgent[Question, Answer](config=AgentConfig(client=<instructor_client>, model="<model>"))
print(agent.run(Question(text="17 * 23?")).value)
```

## 4. TypeScript / JavaScript frameworks

Many agent products are web apps, so the JavaScript world has its own strong options. These are shown as code only (this is a Python notebook).

| Framework | What it is | When to use | Notes (late 2026) |
|---|---|---|---|
| **Mastra** (`@mastra/core`) | TypeScript-first framework: agents, tools, **workflows** (graph with `.then()`, `.branch()`, suspend/resume), memory, RAG, evals, a local dev playground, MCP and A2A | full-stack TS teams who want "LangGraph + CrewAI ideas" natively in TypeScript | stable **1.0** released January 2026 |
| **Vercel AI SDK** (`ai`) | the most-used TS library for calling models and streaming to UIs; v6 added a first-class **`ToolLoopAgent`** class with stop conditions and tool approval | Next.js / React apps, streaming chat UIs, simple agents | v6 replaced the earlier `Experimental_Agent`; newer major versions may rename things, so check the docs |
| **LangGraph.js** (`@langchain/langgraph`) | the JavaScript port of LangGraph: same graph, state, checkpoint and interrupt ideas | teams that want LangGraph but deploy on Node/edge | see the Python [LangGraph notebook](12_langgraph_hands_on.ipynb) for the concepts |
| **OpenAI Agents SDK (JS)** (`@openai/agents`) | the TypeScript twin of section 3.2: agents, handoffs, guardrails, tracing, realtime voice agents | OpenAI-centric TS backends and voice apps | some newer Python features (e.g. sandbox agents) reached TS later |

```typescript
// Mastra
import { Agent } from "@mastra/core/agent";
import { createTool } from "@mastra/core/tools";
import { z } from "zod";
const multiply = createTool({
  id: "multiply", description: "Multiply two numbers",
  inputSchema: z.object({ a: z.number(), b: z.number() }),
  execute: async ({ context }) => ({ result: context.a * context.b }),
});
const calculator = new Agent({ name: "calculator", instructions: "Use tools for maths.",
                               model: "<provider/model>", tools: { multiply } });
const res = await calculator.generate("How many pencils are in 17 boxes of 23?");

// Vercel AI SDK (v6 style)
import { ToolLoopAgent, tool, stepCountIs } from "ai";
const agent = new ToolLoopAgent({
  model: "<provider/model>", instructions: "Use tools for maths.",
  tools: { multiply: tool({ description: "Multiply", inputSchema: z.object({ a: z.number(), b: z.number() }),
                            execute: async ({ a, b }) => a * b }) },
  stopWhen: stepCountIs(5),
});
const { text } = await agent.generate({ prompt: "17 boxes of 23 pencils?" });

// OpenAI Agents SDK (JS)
import { Agent, run, tool as oaTool } from "@openai/agents";
const mult = oaTool({ name: "multiply", description: "Multiply", parameters: z.object({ a: z.number(), b: z.number() }),
                      execute: async ({ a, b }) => String(a * b) });
const result = await run(new Agent({ name: "Calculator", instructions: "Use tools for maths.", tools: [mult] }),
                         "17 boxes of 23 pencils?");
```

## 5. Managed / cloud agent platforms

**What they are.** Instead of running the agent loop on your own server, a cloud provider hosts it for you, plus the boring but important parts: identity and permissions, memory storage, tool gateways, scaling, logging, evaluation and governance.

**Analogy.** Running your own web server vs using a managed hosting service. Frameworks (section 3) are the code; platforms are where that code (or a no-code agent) runs.

**When to use / not.** Use a platform when you are already on that cloud, need enterprise security and compliance, or do not want to operate agent infrastructure. Avoid if you need to stay cloud-neutral, run fully on-premises, or keep costs very predictable at small scale. Most platforms now host agents built with **open-source frameworks**, which reduces lock-in.

| Platform (late 2026 name) | Formerly / related | What it gives you | Works with |
|---|---|---|---|
| **Amazon Bedrock AgentCore** | the older **Bedrock Agents** service (now called "classic"; AWS steers new builds to AgentCore) | Runtime (serverless hosting, long sessions), Memory, Identity, Gateway (turn APIs/Lambdas into MCP tools), Browser and Code Interpreter tools, Observability, Evaluations, and a managed "harness" agent loop | Strands, LangGraph, CrewAI, LlamaIndex, OpenAI Agents SDK, Google ADK and others; any model |
| **Microsoft Foundry Agent Service** | **Azure AI Foundry Agent Service** / Azure AI Agent Service (Azure AI Foundry was renamed **Microsoft Foundry**) | hosted agents with threads/conversations, built-in tools (search, code interpreter, file search, Bing grounding), connected agents, multi-agent workflows, enterprise identity and content safety | Microsoft Agent Framework, Semantic Kernel, plus hosted agents from other frameworks; many models incl. OpenAI, Claude and open models |
| **Gemini Enterprise Agent Platform** (Agent Engine / Agent Runtime) | **Vertex AI Agent Engine** (Vertex AI was rebranded in April 2026) | managed runtime for agents, sessions and Memory Bank, evaluation, A2A, an agent gallery / registry for enterprises | Google ADK first, plus LangGraph, LangChain, CrewAI, AG2, LlamaIndex and custom code |
| **OpenAI platform** (Responses API + Agents SDK, Agent Builder, ChatKit) | the **Assistants API** (being retired in favour of the Responses API) | the Responses API runs hosted tools (web search, file search, code interpreter, computer use, remote MCP) server-side, keeps conversation state, and traces runs; visual builders and embeddable chat UI on top | OpenAI models first; Agents SDK for the code side |
| **Anthropic Claude platform** | (new) | Claude API tool use; **Claude Managed Agents** host the agent loop plus a per-session sandbox; the Claude Agent SDK for self-hosted harnesses | Claude models |

**Rule of thumb.** Write your agent with an open framework you can run locally (so you can test it, like in this notebook), then deploy it on the platform of the cloud you already use.

## 6. The same task in N frameworks

Part 1 (the calculator) already ran in sections 3.x. Part 2 is a **two-agent handoff**: a *researcher* collects notes with a `lookup_notes` tool, then a *writer* turns them into a paragraph. This is where frameworks differ most, because each has its own idea of "multi-agent".

| Framework | How "researcher -> writer" is expressed |
|---|---|
| plain Python | call function A, pass its output to function B |
| OpenAI Agents SDK | **handoff**: the researcher's model calls `transfer_to_writer`; the writer takes over the same conversation |
| AutoGen AgentChat | **team**: `RoundRobinGroupChat([researcher, writer])` with a termination condition |
| Microsoft Agent Framework | **orchestration**: `SequentialBuilder(participants=[researcher, writer])` builds a workflow |
| LlamaIndex | **AgentWorkflow** with `can_handoff_to=["writer"]`; the model calls a built-in `handoff` tool |
| PydanticAI | **programmatic hand-off**: ordinary Python passes agent A's typed output to agent B |

The toy "knowledge base" and the scripted writer are shared:

In [20]:
NOTES_DB = {   # tiny toy knowledge base the researcher's tool reads from
    "agent frameworks": ["most frameworks wrap the same model-tool loop",
                         "they differ in orchestration, state and tracing",
                         "pick by language, cloud and how much control you need"],
}
TOPIC = "agent frameworks"

def lookup_notes(topic: str) -> str:
    '''Look up research notes about a topic.'''
    return "; ".join(NOTES_DB.get(topic, ["no notes found"]))

def write_paragraph(notes_text: str) -> str:
    '''What our scripted "writer model" produces from whatever notes it received.'''
    first = notes_text.split(";")[0].strip()
    return f"ARTICLE: In short, {first}. ({len(notes_text.split(';'))} notes used)"

# plain Python baseline: researcher -> writer is just function composition
notes = lookup_notes(TOPIC)
article = write_paragraph(notes)
print("notes  :", notes)
print("article:", article)
RESULTS.append({"framework": "plain Python (no framework)", "task": "research -> write", "status": "ran", "final answer": article})

notes  : most frameworks wrap the same model-tool loop; they differ in orchestration, state and tracing; pick by language, cloud and how much control you need
article: ARTICLE: In short, most frameworks wrap the same model-tool loop. (3 notes used)


In [21]:
async def openai_agents_handoff():
    from agents import Agent, Runner, function_tool, set_tracing_disabled
    from agents.testing import ScriptedModel, ModelStep, assistant_message, function_call
    set_tracing_disabled(True)
    lookup = function_tool(lookup_notes)

    def writer_reply(call):   # read the conversation the writer received and write from the notes in it
        notes_seen = [it["output"] for it in call.input if isinstance(it, dict) and it.get("type") == "function_call_output"
                      and "loop" in str(it.get("output"))]
        return [assistant_message(write_paragraph(notes_seen[-1]))]

    writer = Agent(name="Writer", instructions="Write one paragraph from the notes.",
                   model=ScriptedModel([ModelStep.respond(writer_reply)]))
    researcher = Agent(name="Researcher", instructions="Collect notes, then hand off to the writer.",
                       tools=[lookup], handoffs=[writer],
                       model=ScriptedModel([[function_call("lookup_notes", {"topic": TOPIC}, call_id="r1")],
                                            [function_call("transfer_to_writer", {}, call_id="h1")]]))
    result = await Runner.run(researcher, f"Research {TOPIC}, then write it up.")
    print(f"  started with: Researcher, finished with: {result.last_agent.name}")
    return result.final_output

await run_demo("OpenAI Agents SDK", "research -> write", openai_agents_handoff)

  started with: Researcher, finished with: Writer
[OpenAI Agents SDK] final answer -> ARTICLE: In short, most frameworks wrap the same model-tool loop. (3 notes used)


In [22]:
async def autogen_team():
    from autogen_agentchat.agents import AssistantAgent
    from autogen_agentchat.teams import RoundRobinGroupChat
    from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
    from autogen_ext.models.replay import ReplayChatCompletionClient

    notes_text = lookup_notes(TOPIC)   # replay clients cannot call tools here, so we pre-fill the researcher's reply
    researcher = AssistantAgent("researcher", model_client=ReplayChatCompletionClient([f"NOTES: {notes_text}"]),
                                system_message="Collect notes.")
    writer = AssistantAgent("writer", model_client=ReplayChatCompletionClient([write_paragraph(notes_text) + " DONE"]),
                            system_message="Write one paragraph, then say DONE.")
    team = RoundRobinGroupChat([researcher, writer],                      # speakers take turns in this order
                               termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(6))
    result = await team.run(task=f"Research {TOPIC}, then write it up.")
    for m in result.messages:
        print(f"  {m.source:10s}: {m.content[:90]}")
    print(f"  stop reason: {result.stop_reason}")
    return result.messages[-1].content

await run_demo("AutoGen AgentChat", "research -> write", autogen_team)

  user      : Research agent frameworks, then write it up.
  researcher: NOTES: most frameworks wrap the same model-tool loop; they differ in orchestration, state 
  writer    : ARTICLE: In short, most frameworks wrap the same model-tool loop. (3 notes used) DONE
  stop reason: Text 'DONE' mentioned
[AutoGen AgentChat] final answer -> ARTICLE: In short, most frameworks wrap the same model-tool loop. (3 notes used) DONE


In [23]:
async def maf_sequential():
    from agent_framework import Agent, BaseChatClient, ChatResponse, Message, FunctionInvocationLayer
    from agent_framework.orchestrations import SequentialBuilder

    class FunctionClient(FunctionInvocationLayer, BaseChatClient):
        '''Stand-in model whose reply is computed by a Python function of the conversation.'''
        def __init__(self, fn, **kw):
            super().__init__(**kw); self.fn = fn
        async def _reply(self, messages):
            return ChatResponse(messages=[Message(role="assistant", contents=[self.fn(messages)])])
        def _inner_get_response(self, *, messages, stream, options, **kwargs):
            return self._reply(messages)

    researcher = Agent(client=FunctionClient(lambda msgs: "NOTES: " + lookup_notes(TOPIC)),
                       name="researcher", instructions="Collect notes.")
    writer = Agent(client=FunctionClient(lambda msgs: write_paragraph(msgs[-1].text.replace("NOTES: ", ""))),
                   name="writer", instructions="Write one paragraph from the notes.")   # reads the researcher's message
    workflow = SequentialBuilder(participants=[researcher, writer]).build()
    result = await workflow.run(f"Research {TOPIC}, then write it up.")
    final = result.get_outputs()[-1]
    return final.messages[-1].text

await run_demo("Microsoft Agent Framework", "research -> write", maf_sequential)

[Microsoft Agent Framework] final answer -> ARTICLE: In short, most frameworks wrap the same model-tool loop. (3 notes used)


In [24]:
async def llamaindex_agent_workflow():
    from llama_index.core.llms.mock import MockFunctionCallingLLM
    from llama_index.core.llms import ChatMessage, MessageRole
    from llama_index.core.base.llms.types import ToolCallBlock
    from llama_index.core.agent.workflow import FunctionAgent, AgentWorkflow, ToolCallResult

    def researcher_script(messages, **kw):
        tool_msgs = [m for m in messages if m.role == MessageRole.TOOL]
        if not tool_msgs:     # 1) look up notes   2) hand off, passing the notes as the reason
            return ChatMessage(role="assistant", blocks=[ToolCallBlock(tool_call_id="r1", tool_name="lookup_notes", tool_kwargs={"topic": TOPIC})])
        return ChatMessage(role="assistant", blocks=[ToolCallBlock(tool_call_id="h1", tool_name="handoff",
                           tool_kwargs={"to_agent": "writer", "reason": tool_msgs[-1].content})])

    def writer_script(messages, **kw):
        # the writer sees the shared conversation, including the researcher's lookup_notes result
        notes_text = next(str(m.content) for m in messages
                          if m.role == MessageRole.TOOL and "now handling" not in str(m.content))
        return ChatMessage(role="assistant", content=write_paragraph(notes_text))

    researcher = FunctionAgent(name="researcher", description="Finds notes.", tools=[lookup_notes],
                               llm=MockFunctionCallingLLM(response_generator=researcher_script), can_handoff_to=["writer"])
    writer = FunctionAgent(name="writer", description="Writes a paragraph.", tools=[],
                           llm=MockFunctionCallingLLM(response_generator=writer_script))
    workflow = AgentWorkflow(agents=[researcher, writer], root_agent="researcher")
    handler = workflow.run(user_msg=f"Research {TOPIC}, then write it up.")
    async for ev in handler.stream_events():                    # watch the events as they happen
        if isinstance(ev, ToolCallResult):
            print(f"  [{ev.tool_name}] -> {str(ev.tool_output)[:80]}")
    return str(await handler)

await run_demo("LlamaIndex", "research -> write", llamaindex_agent_workflow)

  [lookup_notes] -> most frameworks wrap the same model-tool loop; they differ in orchestration, sta
  [handoff] -> Agent writer is now handling the request due to the following reason: most frame
[LlamaIndex] final answer -> ARTICLE: In short, most frameworks wrap the same model-tool loop. (3 notes used)


In [25]:
async def pydantic_ai_handoff():
    from pydantic import BaseModel
    from pydantic_ai import Agent
    from pydantic_ai.models.function import FunctionModel
    from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, ToolReturnPart, UserPromptPart

    class Notes(BaseModel):
        points: list[str]

    researcher = Agent(output_type=Notes, tools=[lookup_notes], instructions="Collect notes.")
    def researcher_model(messages, info):
        for part in messages[-1].parts:
            if isinstance(part, ToolReturnPart):        # got notes -> return them as the typed output
                return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name,
                                                         {"points": part.content.split("; ")})])
        return ModelResponse(parts=[ToolCallPart("lookup_notes", {"topic": TOPIC})])

    writer = Agent(instructions="Write one paragraph from the notes.")
    def writer_model(messages, info):
        prompt = next(p.content for m in messages for p in m.parts if isinstance(p, UserPromptPart))
        return ModelResponse(parts=[TextPart(write_paragraph(prompt))])

    notes = (await researcher.run(f"Research {TOPIC}", model=FunctionModel(researcher_model))).output
    print(f"  researcher returned a typed object: {type(notes).__name__} with {len(notes.points)} points")
    article = await writer.run("; ".join(notes.points), model=FunctionModel(writer_model))   # plain Python hand-off
    return article.output

await run_demo("PydanticAI", "research -> write", pydantic_ai_handoff)

  researcher returned a typed object: Notes with 3 points


[PydanticAI] final answer -> ARTICLE: In short, most frameworks wrap the same model-tool loop. (3 notes used)


### What ran?

Every demo recorded itself in `RESULTS`. The table below shows what actually ran in **your** environment. If you only have the base repo packages installed, you will see "skipped" rows; install the optional packages from the top of the notebook to run them all.

In [26]:
results = pd.DataFrame(RESULTS)
print(results["status"].value_counts().to_string(), "\n")
results

status
ran    23 



,framework,task,status,final answer
0,plain Python (no framework),calculator,ran,That is 391 pencils.
1,AutoGen AgentChat,calculator,ran,That is 391 pencils.
2,Microsoft Agent Framework,calculator,ran,That is 391.0 pencils.
3,OpenAI Agents SDK,calculator,ran,That is 391 pencils.
4,OpenAI Agents SDK,guardrail + session,ran,Your name is Ana.
5,Google ADK,calculator,ran,That is 391 pencils.
6,PydanticAI,calculator,ran,That is 391.0 pencils.
7,PydanticAI,structured output,ran,PencilOrder(total=391)
8,smolagents,calculator (CodeAgent),ran,That is 391 pencils.
9,smolagents,calculator (ToolCallingAgent),ran,That is 391 pencils.


In [27]:
# A "vocabulary translation" table: the same idea has a different name in every framework.
vocab = pd.DataFrame([
    ["plain Python",        "function + loop",            "Python function",                  "call B with A's output",            "your own list"],
    ["OpenAI Agents SDK",   "Agent + Runner.run",         "@function_tool",                   "handoffs=[...]",                    "Session"],
    ["Microsoft Agent Fw",  "Agent(client=...)",          "@tool",                            "Sequential/Handoff/GroupChat builders", "threads / sessions, checkpoints"],
    ["AutoGen AgentChat",   "AssistantAgent",             "plain function",                   "RoundRobin / Selector / Swarm team", "model_context"],
    ["Google ADK",          "LlmAgent + Runner",          "plain function / FunctionTool",    "sub_agents, AgentTool, A2A",        "SessionService, MemoryService"],
    ["PydanticAI",          "Agent.run",                  "@agent.tool",                      "delegation via tools / plain Python", "message_history, durable exec"],
    ["smolagents",          "CodeAgent / ToolCallingAgent", "@tool",                          "managed_agents=[...]",              "agent.memory"],
    ["LlamaIndex",          "FunctionAgent",              "plain function / FunctionTool",    "AgentWorkflow(can_handoff_to)",     "Context, Memory"],
    ["Haystack",            "Agent component",            "Tool / create_tool_from_function", "pipelines of agents",               "state schema"],
    ["Agno",                "Agent.run",                  "plain function / Toolkit",         "Team(members=[...])",               "db + memory"],
    ["Strands",             "Agent(...)(prompt)",         "@tool",                            "agents-as-tools, Swarm, Graph",     "session manager"],
    ["DSPy",                "dspy.ReAct / Module",        "plain function / dspy.Tool",       "compose modules in Python",         "dspy.History"],
    ["LangGraph (nb 12)",   "StateGraph.compile()",       "@tool + ToolNode",                 "edges / Command(goto=...)",         "checkpointer"],
    ["CrewAI (nb 13)",      "Crew.kickoff",               "@tool / BaseTool",                 "Process.sequential / hierarchical", "memory=True"],
], columns=["framework", "an agent is...", "a tool is...", "multi-agent via...", "state / memory via..."])
vocab

,framework,an agent is...,a tool is...,multi-agent via...,state / memory via...
0,plain Python,function + loop,Python function,call B with A's output,your own list
1,OpenAI Agents SDK,Agent + Runner.run,@function_tool,handoffs=[...],Session
2,Microsoft Agent Fw,Agent(client=...),@tool,Sequential/Handoff/GroupChat builders,"threads / sessions, checkpoints"
3,AutoGen AgentChat,AssistantAgent,plain function,RoundRobin / Selector / Swarm team,model_context
4,Google ADK,LlmAgent + Runner,plain function / FunctionTool,"sub_agents, AgentTool, A2A","SessionService, MemoryService"
5,PydanticAI,Agent.run,@agent.tool,delegation via tools / plain Python,"message_history, durable exec"
6,smolagents,CodeAgent / ToolCallingAgent,@tool,managed_agents=[...],agent.memory
7,LlamaIndex,FunctionAgent,plain function / FunctionTool,AgentWorkflow(can_handoff_to),"Context, Memory"
8,Haystack,Agent component,Tool / create_tool_from_function,pipelines of agents,state schema
9,Agno,Agent.run,plain function / Toolkit,Team(members=[...]),db + memory


## 7. The big comparison table

Read this as a **map, not a ranking**. "Maturity" is a rough judgement of stability and production use as of late 2026, not a measured score. "MCP / A2A: yes" means an official or first-party integration exists; check the docs for details.

| Framework | Language | Abstraction level | Multi-agent style | State / memory | Human-in-the-loop | Tracing | MCP / A2A | Maturity (late 2026) | Best for |
|---|---|---|---|---|---|---|---|---|---|
| **LangGraph** (nb 12) | Py, TS | low (you draw the graph) | graph, supervisor, swarm | checkpointers, stores, time travel | interrupts, first-class | LangSmith, OTel | MCP yes / A2A via server | high | complex, durable, controllable flows |
| **CrewAI** (nb 13) | Py | high (roles, crews) | role-based crews, flows | built-in memory | human input on tasks | integrations, OTel | MCP yes / A2A yes | high | fast role-based prototypes and business automations |
| **Microsoft Agent Framework** | Py, .NET | medium | sequential, concurrent, group chat, handoff, Magentic, workflows | threads, checkpoints | workflow request/response, approvals | OTel built in | MCP yes / A2A yes | medium-high (1.0 in 2026) | Azure / .NET shops, AutoGen + SK migrations |
| **AutoGen 0.4+** | Py, .NET | medium | group chats, Magentic-One | model context, save/load state | user proxy agent | OTel | MCP yes / A2A no | maintenance mode | existing AutoGen code; research |
| **AG2** | Py | medium | conversations, group chat, swarm | chat history | human_input_mode | OTel / logging | MCP yes / A2A beta | medium (heading to 1.0) | classic AutoGen-style conversations |
| **Semantic Kernel** | C#, Py, Java | medium | agent group chat (moving to MAF) | chat history, memory connectors | via filters | OTel | MCP yes / A2A samples | high but superseded for agents | existing SK enterprise apps |
| **OpenAI Agents SDK** | Py, TS | low-medium | handoffs, agents-as-tools | sessions | tool approval, guardrails | built-in traces, OTel | MCP yes / A2A no | high | OpenAI-centric apps, voice, sandboxed long tasks |
| **Google ADK** | Py, Java, Go, TS | medium | sub-agents, sequential/parallel/loop, graph workflows (2.0) | session, memory, artifact services | workflow HITL, tool confirmation | OTel, Cloud Trace | MCP yes / A2A yes | medium-high | Gemini / Google Cloud, A2A |
| **Claude Agent SDK** | Py, TS | high (ready harness) | subagents | sessions, context compaction | permissions, hooks | OTel | MCP yes (core) / A2A no | high | coding, file and ops agents on Claude |
| **PydanticAI** | Py | low-medium | delegation, graphs (pydantic-graph) | message history, durable exec (Temporal etc.) | deferred tools / approvals | Logfire, OTel | MCP yes / A2A add-on | high | typed, testable agents with structured output |
| **smolagents** | Py | low | managed agents | memory steps | step callbacks | OTel | MCP yes / A2A no | medium | code-acting agents, open models, learning |
| **LlamaIndex Workflows** | Py, TS | medium | AgentWorkflow handoffs | Context, memory | wait-for-event (HITL events) | OTel, partners | MCP yes / A2A via adapters | high | document / RAG-heavy agents |
| **Haystack** | Py | medium (pipelines) | agents inside pipelines | agent state schema | breakpoints, confirmation strategies | OTel, Langfuse etc. | MCP yes / A2A no | high | production RAG/search pipelines |
| **Agno** | Py | high | teams (coordinate / route / broadcast / tasks), workflows | db-backed memory and storage | user confirmation on tools | AgentOS, OTel | MCP yes / A2A via AgentOS | medium-high | fast full-stack agents with memory and an API |
| **Strands Agents** | Py, TS | low (model-driven) | agents-as-tools, swarm, graph, workflow | session managers | interrupts / approvals | OTel | MCP yes / A2A yes | medium-high | AWS, minimal model-driven agents |
| **DSPy** | Py | different (declarative program) | compose modules | dspy.History | not a focus | MLflow, callbacks | MCP tools convertible / A2A no | high (research + prod) | measurable quality, prompt optimisation |
| **Deep Agents** | Py, TS | high (harness on LangGraph) | sub-agents | virtual files + LangGraph | LangGraph interrupts | LangSmith | MCP via adapters | medium | long research/coding tasks on LangGraph |
| **Letta** | Py, TS clients | high (agent server) | multi-agent via tools / shared memory | core + archival memory, persistent | approvals on tools | built-in ADE, OTel | MCP yes | medium | agents that remember users for weeks |
| **CAMEL** | Py | medium | role-playing, workforce societies | memory modules | human toolkits | logging | MCP yes | medium (research) | research, synthetic data |
| **MetaGPT / ChatDev** | Py | high (SOP roles) | software-company roles | shared message pool | limited | logging | limited | research | studying structured role pipelines |
| **BeeAI** | Py, TS | medium | workflows, requirement agents | memory modules | yes | OTel | MCP yes / A2A yes | medium | open-governance (Linux Foundation) projects |
| **Langroid** | Py | medium | message-passing tasks | agent state | human-in-loop tasks | logging | MCP yes | medium | simple actor-style multi-agent code |
| **Atomic Agents** | Py | low (typed bricks) | explicit chaining | context providers | your code | your code | MCP via tools | medium | predictable, schema-first pipelines |
| **OpenAI Swarm** | Py | very low | handoffs | none | none | none | no | **superseded** (educational) | reading the source to learn |
| **Mastra** | TS | medium | agent networks, workflows | memory, storage | suspend/resume | OTel, playground | MCP yes / A2A yes | medium-high (1.0) | full-stack TypeScript teams |
| **Vercel AI SDK** | TS | low | tool loops, sub-agents as tools | your code / UI state | tool approval | OTel | MCP client yes | high | streaming chat UIs and simple TS agents |

**How to read "abstraction level".** *Low* = you write more code and keep more control (LangGraph, PydanticAI, Strands, smolagents). *High* = the framework decides more for you (CrewAI, Agno, Claude Agent SDK, Deep Agents, Letta). Neither is better; it is a trade-off between **speed of the first demo** and **control of the 100th edge case**.

## 8. How to choose

Start from your constraints (language, cloud, model), then from the shape of the work.

```mermaid
flowchart TD
    A[Start: what are you building?] --> B{One or two fixed model calls?}
    B -->|yes| P[No framework: plain code + model SDK]
    B -->|no| C{Main language?}
    C -->|TypeScript| T{UI-heavy web app?}
    T -->|yes| V[Vercel AI SDK]
    T -->|no, need workflows + memory| MA[Mastra or LangGraph.js]
    C -->|Python / .NET| D{Locked to a cloud or model vendor?}
    D -->|Azure / .NET| MAF[Microsoft Agent Framework]
    D -->|AWS| ST[Strands + AgentCore]
    D -->|Google Cloud / Gemini| ADK[Google ADK]
    D -->|OpenAI-centric| OAI[OpenAI Agents SDK]
    D -->|Claude, coding or file agents| CAS[Claude Agent SDK]
    D -->|no lock-in| E{What matters most?}
    E -->|explicit control, durable state, HITL| LG[LangGraph]
    E -->|typed outputs and tests| PA[PydanticAI]
    E -->|team of roles, fast prototype| CR[CrewAI]
    E -->|documents and RAG| LI[LlamaIndex or Haystack]
    E -->|long-term memory per user| LE[Letta]
    E -->|measurable quality, prompt optimisation| DS[DSPy]
    E -->|small, code-writing agent| SM[smolagents]
```

**Five questions to ask before picking:**
1. **Language and team skills.** Python, TypeScript or .NET?
2. **Cloud and model.** Will this live on Azure, AWS, Google Cloud, or stay neutral? Is one model vendor a given?
3. **Control vs speed.** Do you need to draw and inspect every step (graph), or is "let the model plan" fine?
4. **Production needs.** Durable state, human approval, tracing, evaluation, security reviews?
5. **Ecosystem and maturity.** Is the project actively maintained? Does it have an exit path (MCP tools and A2A make switching easier)?

Below, a tiny chooser that filters a small table of frameworks by your needs. It is a teaching toy that encodes the flowchart above, not a scientific ranking.

In [28]:
CHOICES = pd.DataFrame([
    # name,                   languages,         cloud,     traits (rough, from section 7)
    ("LangGraph",                 {"py", "ts"},        "neutral", {"graph", "durable", "hitl", "control"}),
    ("CrewAI",                    {"py"},              "neutral", {"roles", "fast-prototype"}),
    ("Microsoft Agent Framework", {"py", "dotnet"},    "azure",   {"graph", "durable", "hitl", "multi-agent", "enterprise"}),
    ("OpenAI Agents SDK",         {"py", "ts"},        "openai",  {"handoffs", "guardrails", "fast-prototype", "voice"}),
    ("Google ADK",                {"py", "ts", "java", "go"}, "google", {"multi-agent", "a2a", "graph", "enterprise"}),
    ("Strands Agents",            {"py", "ts"},        "aws",     {"minimal", "multi-agent", "a2a"}),
    ("Claude Agent SDK",          {"py", "ts"},        "anthropic", {"coding", "files", "long-tasks"}),
    ("PydanticAI",                {"py"},              "neutral", {"typed", "testing", "minimal", "durable"}),
    ("smolagents",                {"py"},              "neutral", {"minimal", "code-actions", "open-models"}),
    ("LlamaIndex",                {"py", "ts"},        "neutral", {"rag", "events"}),
    ("Haystack",                  {"py"},              "neutral", {"rag", "pipelines", "enterprise"}),
    ("Agno",                      {"py"},              "neutral", {"fast-prototype", "memory", "serving"}),
    ("DSPy",                      {"py"},              "neutral", {"optimization", "evaluation"}),
    ("Letta",                     {"py", "ts"},        "neutral", {"memory", "long-lived"}),
    ("Mastra",                    {"ts"},              "neutral", {"graph", "memory", "fast-prototype"}),
    ("Vercel AI SDK",             {"ts"},              "neutral", {"ui-streaming", "minimal"}),
], columns=["framework", "languages", "home cloud", "traits"])

def recommend(language, needs, cloud=None, top=3):
    '''Keep frameworks in your language; score = how many of your needs they match (+1 if they match your cloud).'''
    df = CHOICES[CHOICES["languages"].apply(lambda langs: language in langs)].copy()
    df["score"] = df["traits"].apply(lambda t: len(t & set(needs)))
    if cloud:
        df["score"] += (df["home cloud"] == cloud).astype(int)
    df = df[df["score"] > 0].sort_values("score", ascending=False)
    return df.head(top)[["framework", "home cloud", "score"]].reset_index(drop=True)

print("Python, needs durable state + human approval, no cloud preference:")
print(recommend("py", {"durable", "hitl"}).to_string(), "\n")
print("Python on AWS, wants something minimal with multi-agent:")
print(recommend("py", {"minimal", "multi-agent"}, cloud="aws").to_string(), "\n")
print("TypeScript web app with streaming UI:")
print(recommend("ts", {"ui-streaming", "minimal"}).to_string())

Python, needs durable state + human approval, no cloud preference:
                   framework home cloud  score
0                  LangGraph    neutral      2
1  Microsoft Agent Framework      azure      2
2                 PydanticAI    neutral      1 

Python on AWS, wants something minimal with multi-agent:
                   framework home cloud  score
0             Strands Agents        aws      3
1  Microsoft Agent Framework      azure      1
2                 Google ADK     google      1 

TypeScript web app with streaming UI:
        framework home cloud  score
0   Vercel AI SDK    neutral      2
1  Strands Agents        aws      1


**What just happened?** The chooser kept only frameworks in your language, then counted matching traits. Real decisions also weigh team experience, licences, security reviews and support, which no table can capture. Try your own needs, e.g. `recommend("py", {"rag", "enterprise"})`.

### Common pitfalls when picking a framework

| Pitfall | Why it hurts | What to do |
|---|---|---|
| Picking by GitHub stars or hype | popular is not the same as right for your constraints | answer the five questions above first |
| Learning from old tutorials | many APIs changed in 2025-2026 (AutoGen 0.2 -> 0.4 -> MAF, Phidata -> Agno, Swarm -> Agents SDK, Vertex AI -> Gemini Enterprise Agent Platform) | check the date on any tutorial; prefer official docs |
| Too much abstraction too early | high-level frameworks hide prompts and loops, which makes debugging hard | start small; make sure you can see every prompt and tool call (tracing) |
| Ignoring telemetry defaults | some frameworks send usage pings or upload traces by default | read the "telemetry / tracing" page of the docs (see Setup cell) |
| Lock-in through tools | tools written in one framework's format are hard to move | write tools as plain functions or **MCP servers**; expose agents via **A2A** |
| No tests | agents are non-deterministic | use the official fake/test models (as in this notebook) for unit tests, and evals for quality |

## 9. Recap / cheat sheet

| If you remember one line per framework... | |
|---|---|
| **plain code** | a 20-line loop; enough for simple tasks |
| **LangGraph** | draw the graph yourself; best control and durability (nb 12) |
| **CrewAI** | describe roles and tasks; fastest "team of agents" (nb 13) |
| **Microsoft Agent Framework** | AutoGen + Semantic Kernel merged; Python and .NET; Azure-friendly |
| **AutoGen / AG2** | AutoGen 0.4 is in maintenance; AG2 is the community fork of 0.2 |
| **Semantic Kernel** | enterprise plugins SDK; new agent work goes to Agent Framework |
| **OpenAI Agents SDK** | agents, handoffs, guardrails, sessions, tracing; successor of Swarm |
| **Google ADK** | Google's full toolchain; workflow agents, A2A, Gemini-first |
| **Claude Agent SDK** | the Claude Code harness as a library: built-in file/shell tools, hooks, subagents |
| **PydanticAI** | type-safe agents with validated outputs and great test models |
| **smolagents** | tiny; CodeAgent writes Python as its actions (sandbox it) |
| **LlamaIndex** | event-driven Workflows + AgentWorkflow; strongest for RAG |
| **Haystack** | explicit component pipelines; production search/RAG |
| **Agno** | formerly Phidata; agents, teams, workflows, memory, AgentOS server |
| **Strands + AgentCore** | AWS's minimal model-driven SDK + managed runtime |
| **DSPy** | declare signatures, let optimisers write the prompts |
| **Deep Agents** | LangChain's long-task harness (planning, files, sub-agents) |
| **Letta** | formerly MemGPT; long-lived agents with self-edited memory |
| **CAMEL / MetaGPT / ChatDev** | research-style role-playing societies |
| **BeeAI / Langroid / Atomic Agents** | open-governance / message-passing / typed-bricks alternatives |
| **Swarm** | educational only; superseded |
| **Mastra / Vercel AI SDK / LangGraph.js / OpenAI Agents JS** | the TypeScript options |
| **Managed platforms** | AgentCore (AWS), Foundry Agent Service (Microsoft), Gemini Enterprise Agent Platform (Google), OpenAI Responses/Agents platform, Claude Managed Agents |

**Big ideas:** every framework wraps the same model-tool loop; they differ in orchestration style, state, HITL, tracing and ecosystem. Use official fake models for tests. Keep tools portable (plain functions, MCP) so you can switch.

## 10. Interview questions

**Q1. What do all agent frameworks have in common?**
A loop: call the model, run any tool it asks for, add the result to the conversation, repeat until it answers. Frameworks add orchestration, state, HITL, guardrails and tracing around that loop.

**Q2. What is a handoff, and how is it different from "agent as a tool"?**
In a **handoff**, control moves to another agent, which continues the conversation and gives the final answer. With **agent-as-a-tool**, the first agent calls the second like a function, gets its result back, and stays in charge.

**Q3. AutoGen, AG2 and Microsoft Agent Framework: what is the relationship?**
AG2 is a community fork of AutoGen 0.2. Microsoft rewrote AutoGen as 0.4, then merged its ideas with Semantic Kernel into Microsoft Agent Framework, which is now Microsoft's recommended path; AutoGen itself is in maintenance mode.

**Q4. When would you choose smolagents' CodeAgent over a JSON tool-calling agent?**
When tasks need several tool calls combined with logic (loops, maths, data wrangling), because one code action can do what would take many JSON calls. You must run that code in a sandbox.

**Q5. What makes DSPy different from the other frameworks?**
You declare inputs/outputs (signatures) and a metric; DSPy's optimisers write and tune the prompts and examples for you. It is about programming and optimising LM pipelines, not hand-writing prompts.

**Q6. How do you unit-test an agent without paying for API calls?**
Plug in a fake or scripted model (PydanticAI `TestModel`/`FunctionModel`, AutoGen `ReplayChatCompletionClient`, OpenAI Agents `ScriptedModel`, DSPy `DummyLM`, LlamaIndex `MockFunctionCallingLLM`) and assert that the right tools were called with the right arguments. Use evals with real models separately for quality.

**Q7. What are MCP and A2A, and why do they matter when choosing a framework?**
MCP standardises how agents get **tools** from servers; A2A standardises how **agents talk to other agents** across frameworks and companies. Support for both lowers lock-in: tools and agents can be reused if you switch frameworks.

**Q8. Framework vs managed platform: what is the difference?**
A framework is the code library you use to build the agent. A managed platform (AgentCore, Foundry Agent Service, Gemini Enterprise Agent Platform, OpenAI's platform) hosts and operates agents: scaling, identity, memory, monitoring. Many platforms can host agents built with open frameworks.

**Q9. A team wants "the best framework". What do you ask first?**
Language, cloud/model constraints, how much control and durability they need (HITL, long-running state), production requirements (tracing, security, evals), and the team's experience. Then prototype the riskiest part in one or two candidates.

**Q10. Name two risks of high-level frameworks.**
Hidden prompts and control flow make debugging hard, and fast API churn makes code and tutorials go stale. Mitigate with tracing, pinned versions, and keeping tools framework-independent.

## 11. Further reading

Official documentation (start here; names and URLs as of late 2026):

- **Microsoft Agent Framework**: https://learn.microsoft.com/agent-framework/ and https://github.com/microsoft/agent-framework (includes migration guides from AutoGen and Semantic Kernel)
- **AutoGen** (maintenance mode): https://github.com/microsoft/autogen
- **AG2**: https://docs.ag2.ai and https://github.com/ag2ai/ag2
- **Semantic Kernel**: https://learn.microsoft.com/semantic-kernel/
- **OpenAI Agents SDK**: https://openai.github.io/openai-agents-python/ (JS: https://openai.github.io/openai-agents-js/)
- **OpenAI Swarm** (historical): https://github.com/openai/swarm
- **Google Agent Development Kit**: https://google.github.io/adk-docs/ (also https://adk.dev)
- **Claude Agent SDK**: https://code.claude.com/docs/en/agent-sdk
- **PydanticAI**: https://ai.pydantic.dev
- **smolagents**: https://huggingface.co/docs/smolagents
- **LlamaIndex** (Workflows and agents): https://docs.llamaindex.ai
- **Haystack**: https://docs.haystack.deepset.ai
- **Agno**: https://docs.agno.com
- **Strands Agents**: https://strandsagents.com ; **Amazon Bedrock AgentCore**: https://docs.aws.amazon.com/bedrock-agentcore/
- **DSPy**: https://dspy.ai (paper: Khattab et al., "DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines", arXiv:2310.03714)
- **LangChain Deep Agents**: https://docs.langchain.com (Deep Agents section) and https://github.com/langchain-ai/deepagents
- **Letta**: https://docs.letta.com (paper: Packer et al., "MemGPT: Towards LLMs as Operating Systems", arXiv:2310.08560)
- **CAMEL**: https://docs.camel-ai.org (paper: Li et al., "CAMEL: Communicative Agents for 'Mind' Exploration of Large Language Model Society", arXiv:2303.17760)
- **MetaGPT**: https://github.com/FoundationAgents/MetaGPT (paper: Hong et al., "MetaGPT: Meta Programming for a Multi-Agent Collaborative Framework", arXiv:2308.00352); **ChatDev**: https://github.com/OpenBMB/ChatDev (Qian et al., "ChatDev: Communicative Agents for Software Development", arXiv:2307.07924)
- **BeeAI Framework**: https://framework.beeai.dev and https://github.com/i-am-bee
- **Langroid**: https://langroid.github.io/langroid/
- **Atomic Agents**: https://github.com/BrainBlend-AI/atomic-agents
- **Mastra**: https://mastra.ai/docs ; **Vercel AI SDK**: https://ai-sdk.dev ; **LangGraph.js**: https://langchain-ai.github.io/langgraphjs/
- **Managed platforms**: Microsoft Foundry Agent Service (https://learn.microsoft.com/azure/ai-foundry/agents/), Gemini Enterprise Agent Platform (https://cloud.google.com, formerly Vertex AI Agent Engine), OpenAI platform docs (https://platform.openai.com/docs), Claude platform docs (https://platform.claude.com/docs)
- **Protocols**: Model Context Protocol (https://modelcontextprotocol.io), Agent2Agent protocol (https://a2a-protocol.org)

Related notebooks in this repo: [03_langgraph](03_langgraph.ipynb), [04_multi_agent_frameworks](04_multi_agent_frameworks.ipynb), [05_mcp_and_tools](05_mcp_and_tools.ipynb), [06_agent_observability](06_agent_observability.ipynb), [07_advanced_agent_frameworks](07_advanced_agent_frameworks.ipynb), [11_context_engineering_memory_and_agent_protocols](11_context_engineering_memory_and_agent_protocols.ipynb), [12_langgraph_hands_on](12_langgraph_hands_on.ipynb), [13_crewai_and_multi_agent_frameworks_hands_on](13_crewai_and_multi_agent_frameworks_hands_on.ipynb).